# Integrasi Confident Learning dan CORN pada IndoBERT untuk Prediksi Rating Ulasan Google Play Store

**Notebook final (run murni)**

| | |
| --- | --- |
| Nama | Ryan Besto Saragih |
| NIM | 23051204205 |
| Pembimbing | Anita Qoiriah, S.Kom., M.Kom. |
| Repositori | https://github.com/bestoism/corn-cl-indobert-rating-noise |
| Folder Drive | /content/drive/MyDrive/SKRIPSI_CORN_CL_FINAL |

### Definisi run murni

Pada notebook ini, run murni berarti seluruh artefak komputasi (partisi data, probabilitas proxy,
varian data bersih, model, prediksi, tabel hasil, dan uji statistik) dibangun ulang dari data raw dan
lexicon dalam eksekusi ini. Notebook tidak bergantung pada cache, checkpoint, atau hasil eksekusi
sebelumnya. Cache di dalam satu eksekusi yang sama, yaitu berkas unik yang dipakai untuk melanjutkan
setelah runtime Colab terputus, tetap diperbolehkan. Input manusia (anotasi manual) hanya dipakai
kembali setelah verifikasi hash terhadap hasil run ini berhasil.

Berkas yang dipertahankan di Drive: data raw (all_reviews_master.csv), lexicon, scraping_summary.csv
dan manifest korpus asli (folder preserved), serta berkas anotasi beserta sidecar hash-nya (folder annotations).

### Daftar isi

1. Persiapan lingkungan dan audit keadaan Drive
2. Verifikasi korpus (hash terkunci)
3. Laporan kualitas data dan EDA (Subbab 3.2)
4. Praproses teks (Subbab 3.3)
5. Pembagian data dan bukti nol overlap (Subbab 3.4)
6. Ablasi proxy classifier P1 sampai P4, kalibrasi, dan diagnostik (Subbab 3.5, 3.6)
7. Penetapan P4, deteksi noise, pemilihan metode filter, resolusi konflik, dan tiga varian data (Subbab 3.6, 3.8)
8. Validasi manusia (Subbab 3.7)
9. Pelatihan enam skenario dan tiga seed (Subbab 3.9)
10. Evaluasi dan uji signifikansi H1 sampai H5 (Subbab 3.11, 3.12)
11. Subset uji emas (Subbab 3.10)
12. Sensitivitas jumlah fold K (Subbab 3.5)
13. Sensitivitas partisi data (Subbab 3.4, 3.12)
14. Ringkasan akhir
15. Manifest hasil dan unduhan cadangan

### Petunjuk menjalankan

- Jalankan sel dari atas ke bawah. Setiap bagian diakhiri sel pemeriksaan mandiri yang mencetak
  [OK] atau [BERHENTI]. Bila [BERHENTI], eksekusi dihentikan dan alasannya dicetak. Perbaiki dahulu, jangan dilewati.
- Sel yang menghentikan eksekusi: audit Drive (bagian 1), verifikasi korpus (bagian 2), pemeriksaan
  mandiri tiap bagian, serta pemeriksaan hash anotasi (bagian 8 dan 11).
- Sel yang boleh dilewati: sel pembersihan (bagian 1, dry run sebagai bawaan), sel cadangan zip, dan
  sel penampil tabel.
- Bila runtime putus: jalankan ulang sel konfigurasi (bagian 1.2) dengan LANJUTKAN_EKSEKUSI = True, lalu
  jalankan ulang sel yang terhenti. Unit kerja yang berkasnya sudah terverifikasi akan dilewati.
- Berkas di Drive tidak pernah ditimpa. Bila sebuah berkas sudah ada dengan isi berbeda, eksekusi berhenti.
- Anotasi manual (bagian 8 dan 11) dapat dikerjakan paralel dengan pelatihan. Pelatihan tidak menunggu anotasi.

## 1. Persiapan lingkungan dan audit keadaan Drive

**Tujuan.** Memasang Drive dan repositori, memeriksa versi paket (coral-pytorch minimal 1.3.0), mencatat lingkungan, membuat seluruh folder sekali, dan memastikan Drive berada pada keadaan bersih.

**Rujukan proposal.** Subbab 3.13 (alat dan lingkungan).

**Masukan.** Drive berisi data/raw, lexicon, preserved, dan annotations.

**Keluaran.** logs/environment__{RUN_TAG}.json; seluruh folder keluaran (runs, shared, logs) dibuat sekali.

**Perkiraan waktu GPU.** kurang dari 5 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Audit membandingkan isi Drive dengan daftar yang diizinkan. Pada mode awal, hanya raw, lexicon, preserved, dan annotations yang boleh ada.

**Peringatan.** Bila audit menemukan artefak lain atau nama ganda (misalnya akhiran ' (1)'), eksekusi berhenti. Setelah run dimulai, sel audit hanya lulus pada mode awal bila LANJUTKAN_EKSEKUSI = False dan belum ada artefak run.

### 1.1 Memasang Drive, repositori, dan paket

In [1]:
import os, sys

REPO_DIR   = r"D:\SKRIPSI\corn-cl-indobert-rating-noise"
DATA_ROOT  = r"D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL"
LOCAL_ROOT = r"D:\SKRIPSI\lokal"

os.environ["SKRIPSI_DRIVE_ROOT"] = DATA_ROOT
os.environ["SKRIPSI_LOCAL_ROOT"] = LOCAL_ROOT
os.makedirs(LOCAL_ROOT, exist_ok=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

wajib = [
    os.path.join(DATA_ROOT, "data", "raw", "all_reviews_master.csv"),
    os.path.join(DATA_ROOT, "lexicon", "slang_base.csv"),
]
for p in wajib:
    print(("[OK] " if os.path.isfile(p) else "[BERHENTI] tidak ada: ") + p)
assert all(os.path.isfile(p) for p in wajib), "Berkas wajib belum lengkap"

import torch
print("CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
    print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("[OK] Lingkungan lokal siap.")

[OK] D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\data\raw\all_reviews_master.csv
[OK] D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\lexicon\slang_base.csv
CUDA : True
GPU  : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM : 6.4 GB
[OK] Lingkungan lokal siap.


### 1.2 Konfigurasi eksekusi dan impor

In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

# ---- Pengaturan yang boleh diubah peneliti ----
LANJUTKAN_EKSEKUSI = True        # True hanya untuk melanjutkan eksekusi yang sama setelah runtime putus
HEMAT_SPLIT_SENSITIVITAS = False  # True: satu seed bobot per split sensitivitas (WAJIB dinyatakan sebagai keterbatasan)
ALUR_ANOTASI_VALIDASI = "B"       # "A": pakai kembali anotasi lama setelah verifikasi hash; "B": buat sampel baru
ALUR_ANOTASI_GOLD = "B"           # idem untuk uji emas
SALIN_CHECKPOINT_KE_DRIVE = False # checkpoint tidak perlu disalin ke Drive (prediksi sudah tersimpan)

from src import (config, drive_io, env_log, corpus, quality_report, preprocess, data_split, ablation, clean,
                 proxy, pipeline, human_validation, gold_test, sensitivity_k, significance, consistency,
                 summary, selfcheck, train)

RUN_TAG = pipeline.new_run_tag()
ctx_main = config.RunPaths(config.MAIN_SPLIT_SEED)
SEEDS_MAIN = pipeline.weight_seeds_for(config.MAIN_SPLIT_SEED)
print("RUN_TAG:", RUN_TAG)
print("Drive root:", config.DRIVE_ROOT)

D:\SKRIPSI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_TAG: 20261004_060807
Drive root: D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL


### 1.3 Pemeriksaan versi, audit keadaan Drive, pembuatan folder, dan pencatatan lingkungan

In [3]:
coral_version = env_log.check_coral_version()
print("coral-pytorch:", coral_version)

audit_mode = "berjalan" if LANJUTKAN_EKSEKUSI else "awal"
df_audit, duplicates = drive_io.audit_drive_state(mode=audit_mode)

config.ensure_all_dirs()
env_info, _ = env_log.write_environment_log(RUN_TAG)
display(pd.Series({"python": env_info["python"][:40], "gpu": env_info["gpu"], **env_info["paket"]}))

coral-pytorch: 1.4.0
Folder proyek : D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL
Mode audit    : berjalan
               jumlah  total_mb
kelas                          
DIPERBOLEHKAN      91     22.27
DIPERTAHANKAN      28     12.47

[OK] Audit Drive lulus.


python           3.13.3 (tags/v3.13.3:6280bb5, Apr  8 202
gpu                    NVIDIA GeForce RTX 3060 Laptop GPU
torch                                        2.14.1+cu126
transformers                                       5.18.0
scikit-learn                                        1.9.1
cleanlab                                            2.9.0
coral-pytorch                                       1.4.0
scipy                                              1.18.1
statsmodels                                        0.15.0
numpy                                               2.5.3
pandas                                              3.0.6
matplotlib                                         3.11.2
dtype: str

### 1.4 Pembersihan artefak (opsional, dry run sebagai bawaan)

Sel ini hanya menampilkan daftar yang akan dihapus. Berkas yang dipertahankan (raw, lexicon, preserved,
annotations) tidak pernah dihapus. Penghapusan nyata memerlukan dry_run=False dan konfirmasi persis
'HAPUS SEKARANG'.

In [4]:
akan_dihapus = drive_io.bersihkan_artefak(dry_run=True)

Daftar yang akan dihapus:
  - logs
  - runs
  - shared

[DRY RUN] Tidak ada yang dihapus. Untuk menghapus: dry_run=False dan konfirmasi='HAPUS SEKARANG'.


### 1.5 Pemeriksaan mandiri bagian 1

In [5]:
selfcheck.check_environment()

Pemeriksaan mandiri: 1. Lingkungan dan Drive
  [OK] coral-pytorch 1.4.0 memenuhi syarat minimal 1.3.0
  [OK] GPU CUDA tersedia
  [OK] folder ada: shared
  [OK] folder ada: shared\processed
  [OK] folder ada: shared\results
  [OK] folder ada: shared\results\quality
  [OK] folder ada: shared\results\sensitivity_partisi
  [OK] folder ada: logs
  [OK] folder ada: runs
  [OK] folder ada: annotations
  [OK] folder ada: preserved
[OK] 1. Lingkungan dan Drive: seluruh pemeriksaan umum lulus.


True

## 2. Verifikasi korpus (hash terkunci)

**Tujuan.** Memastikan korpus mentah identik dengan korpus yang dikunci, dengan membandingkan SHA-256 dan jumlah baris terhadap nilai yang tertulis di kode (src/config.py), bukan terhadap manifest yang dibuat ulang oleh run ini.

**Rujukan proposal.** Subbab 3.2 dan Batasan Masalah (penguncian berkas melalui hash SHA-256).

**Masukan.** data/raw/all_reviews_master.csv.

**Keluaran.** shared/results/corpus_verification.json.

**Perkiraan waktu GPU.** kurang dari 1 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Kedua status cocok (SHA-256 dan jumlah baris) harus bernilai True.

**Peringatan.** Scraping tidak boleh dijalankan ulang. Bila hash tidak cocok, eksekusi berhenti karena seluruh eksperimen harus berjalan di atas korpus yang identik.

In [6]:
verif = corpus.verify_master()
display(pd.Series(verif))
corpus.write_verification(verif)

master_file         D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\data\raw\all_...
sha256_dihitung     c589b92cbe2fbc34d67e43a0a866a281cb31603a190af4...
sha256_di_kode      c589b92cbe2fbc34d67e43a0a866a281cb31603a190af4...
sha256_cocok                                                     True
n_baris_dihitung                                                16740
n_baris_di_kode                                                 16740
n_baris_cocok                                                    True
dtype: object

{'path': 'D:\\SKRIPSI\\SKRIPSI_CORN_CL_FINAL\\shared\\results\\corpus_verification.json',
 'status': 'ditulis',
 'bytes': 375,
 'sha256': 'f0c876d523b57e825a4c25c646006115332b27f72eeeeb25f9e7ada3cbb06fe4'}

In [7]:
selfcheck.check_corpus()

Pemeriksaan mandiri: 2. Verifikasi korpus
  [OK] berkas corpus_verification.json ada
  [OK] SHA-256 sama dengan nilai di kode
  [OK] jumlah baris sama dengan nilai di kode
[OK] 2. Verifikasi korpus: seluruh pemeriksaan umum lulus.


True

## 3. Laporan kualitas data dan EDA

**Tujuan.** Menghasilkan karakterisasi korpus berkuota: distribusi rating, distribusi tanggal per aplikasi dan rating (confound temporal), distribusi panjang teks, teks duplikat persis, teks identik dengan rating berbeda, dan simulasi risiko kebocoran pada split acak biasa.

**Rujukan proposal.** Subbab 3.2 (kontrol kualitas data).

**Masukan.** all_reviews_master.csv (seluruhnya dapat dibuat ulang tanpa scraping).

**Keluaran.** shared/results/quality/*.csv dan tiga gambar (fig01 sampai fig03).

**Perkiraan waktu GPU.** tanpa GPU, kurang dari 2 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Tabel tanggal menunjukkan apakah kelas rating jarang (2 dan 3) ditelusuri lebih jauh ke belakang. Tabel panjang teks dan duplikat dipakai untuk memeriksa dugaan penetapan kuota pada Subbab 3.2. Hasil hanya mendeskripsikan data dan tidak mengubah keputusan desain.

**Peringatan.** Jumlah ulasan yang diperiksa dan digugurkan filter bahasa (Bab III) TIDAK dapat dibuat ulang dari master. Angkanya hanya ada pada preserved/scraping_summary.csv.

In [8]:
qr = quality_report.run_quality_report()
display(qr["rating"]); display(qr["per_app"]); display(qr["severity"])
display(qr["length_per_rating"]); display(pd.Series(qr["summary"]))

Baris master: 16740 | rentang tanggal: 2025-08-31 s.d. 2026-09-07


,rating,n,persen
0,1,3779,22.57
1,2,2976,17.78
2,3,2995,17.89
3,4,3176,18.97
4,5,3814,22.78


rating,source_app,1,2,3,4,5
0,Gojek,1225,917,901,1035,1230
1,SeaBank,1342,994,1045,1088,1329
2,Tokopedia,1212,1065,1049,1053,1255


,aplikasi,rating_rentang_terpanjang,rentang_terpanjang_hari,rating_rentang_terpendek,rentang_terpendek_hari,selisih_hari
0,Gojek,4,245,1,44,201
1,SeaBank,2,189,5,4,185
2,Tokopedia,4,372,1,34,338


,count,mean,median,std
rating,,,,
1,3779,21.24,16.0,17.62
2,2976,20.72,16.0,17.59
3,2995,16.65,12.0,16.11
4,3176,10.82,6.0,14.03
5,3814,7.15,4.0,10.79


n_baris                        16740.00
pct_teks_sangat_pendek            24.61
n_grup_duplikat                  201.00
n_baris_duplikat                2832.00
pct_baris_duplikat                16.92
n_grup_konflik                   138.00
n_baris_konflik                 2688.00
pct_baris_konflik                 16.06
n_train_simulasi               13392.00
n_test_simulasi                 3348.00
n_test_teks_juga_di_train        555.00
pct_test_teks_juga_di_train       16.58
dtype: float64

In [9]:
selfcheck.check_quality_report()

Pemeriksaan mandiri: 3. Laporan kualitas data
  [OK] ada dan tidak kosong: 00_ringkasan_kualitas_data.csv
  [OK] ada dan tidak kosong: 01_distribusi_rating_keseluruhan.csv
  [OK] ada dan tidak kosong: 01_distribusi_rating_per_aplikasi.csv
  [OK] ada dan tidak kosong: 02_distribusi_tanggal_per_app_rating.csv
  [OK] ada dan tidak kosong: 03_ringkasan_panjang_teks.csv
  [OK] ada dan tidak kosong: 03_panjang_teks_per_rating.csv
  [OK] ada dan tidak kosong: 04_teks_duplikat_persis_top200.csv
  [OK] ada dan tidak kosong: 07_simulasi_overlap_split_acak_biasa.csv
[OK] 3. Laporan kualitas data: seluruh pemeriksaan umum lulus.


True

## 4. Praproses teks

**Tujuan.** Membersihkan teks sesuai Subbab 3.3: case folding, penghapusan URL dan username, translasi emoji, normalisasi elongasi, normalisasi slang (Kamus Alay dan entri domain), dan penghapusan duplikat murni. Resolusi teks identik dengan rating berbeda sengaja ditunda sampai setelah deteksi noise (bagian 7).

**Rujukan proposal.** Subbab 3.3.

**Masukan.** all_reviews_master.csv dan lexicon/slang_base.csv (serta lexicon/slang_domain.csv bila ada).

**Keluaran.** shared/processed/reviews_clean.csv dan shared/results/preprocessing_summary.csv.

**Perkiraan waktu GPU.** tanpa GPU, beberapa menit (termasuk menghitung tingkat pemotongan token) (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Perhatikan jumlah baris setelah pembersihan, persentase teks sangat pendek, dan persentase ulasan yang terpotong pada 128 token.

**Peringatan.** Kamus tidak diunduh otomatis. Bila lexicon/slang_domain.csv tidak ada, hanya Kamus Alay yang dipakai dan peringatan dicetak (Subbab 3.3 menyebut entri domain dari PRDECT-ID).

In [10]:
if drive_io.is_verified(config.CLEAN_TEXT_FILE):
    df_clean = pd.read_csv(config.CLEAN_TEXT_FILE)
    print(f"[OK] reviews_clean.csv sudah ada: {len(df_clean)} baris.")
else:
    df_clean, prep_summary = preprocess.run_preprocessing()
    display(pd.Series(prep_summary))

[PERINGATAN] Kamus domain tidak ditemukan (D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\lexicon\slang_domain.csv). Proposal Subbab 3.3 menyebut entri domain dari PRDECT-ID; tanpa berkas ini hanya Kamus Alay yang dipakai.
[OK] Kamus dasar dimuat: 15006 entri.
Jumlah data awal: 16740 baris
Cakupan kamus slang (estimasi dari teks mentah): 29410/253442 kata (11.6%)
Teks sangat pendek (tiga kata atau kurang): 24.40%
Teks identik dengan rating berbeda (setelah pembersihan): 151 grup
Resolusi voting mayoritas dijalankan setelah deteksi Confident learning (Subbab 3.3, 3.6).
Setelah dibersihkan: 14221 baris (terbuang: 2519)


D:\SKRIPSI\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Besto\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Ulasan yang terpotong (> 128 token): 0.06%


initial_rows                    16740.00
final_rows                      14221.00
rows_dropped                     2519.00
text_rating_conflict_groups       151.00
pct_very_short_text                24.40
truncation_max_len                128.00
truncation_pct_truncated            0.06
truncation_median_token_len        17.00
slang_total_words              253442.00
slang_normalized_words          29410.00
slang_coverage_pct                 11.60
dtype: float64

In [11]:
selfcheck.check_preprocessing()

Pemeriksaan mandiri: 4. Praproses
  [OK] reviews_clean.csv ada
  [OK] jumlah baris 14221
  [OK] kolom wajib ada
  [OK] tidak ada teks kosong
  [OK] tidak ada duplikat murni
  [OK] review_id unik
[OK] 4. Praproses: seluruh pemeriksaan umum lulus.


True

## 5. Pembagian data dan bukti nol overlap

**Tujuan.** Membagi data 70:10:20 (train, validasi, uji) secara berkelompok berdasarkan teks, stratified menurut rating mayoritas grup, dengan seed partisi 42 (split utama). Mencatat bukti angka bahwa overlap teks dan review_id lintas partisi adalah nol.

**Rujukan proposal.** Subbab 3.4.

**Masukan.** shared/processed/reviews_clean.csv.

**Keluaran.** runs/split_seed42/data/split_train.csv, split_val.csv, split_test.csv, split_summary.csv.

**Perkiraan waktu GPU.** tanpa GPU, kurang dari 1 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Kolom overlap_* pada split_summary.csv harus bernilai 0. Proporsi baris dalam grup teks duplikat dilaporkan.

**Peringatan.** Seed partisi (42, 123, 2024) berbeda dari seed bobot model (42, 123, 2024). Keduanya tidak boleh dicampur.

In [12]:
df_train_main, df_val_main, df_test_main = data_split.create_splits(ctx_main)
display(pd.read_csv(os.path.join(ctx_main.data_dir, "split_summary.csv")).T)

Split seed 42: train=9949, val=1416, test=2856 (dari 14221 baris, 13979 teks unik)
Baris dalam grup teks duplikat: 393 (2.76%)
Overlap lintas partisi (harus nol): {'overlap_teks_train_val': 0, 'overlap_teks_train_test': 0, 'overlap_teks_val_test': 0, 'overlap_review_id_train_val': 0, 'overlap_review_id_train_test': 0, 'overlap_review_id_val_test': 0}


,0
split_seed,42.0000
n_total,14221.0000
n_teks_unik,13979.0000
n_train,9949.0000
n_val,1416.0000
n_test,2856.0000
pct_train,69.9600
pct_val,9.9600
pct_test,20.0800
n_baris_grup_teks_duplikat,393.0000


In [13]:
selfcheck.check_split(ctx_main)

Pemeriksaan mandiri: 5. Pembagian data (split 42)
  [OK] split_summary.csv ada (split 42)
  [OK] jumlah train + val + test sama dengan total
  [OK] overlap_teks_train_val = 0 (harus 0)
  [OK] overlap_teks_train_test = 0 (harus 0)
  [OK] overlap_teks_val_test = 0 (harus 0)
  [OK] overlap_review_id_train_val = 0 (harus 0)
  [OK] overlap_review_id_train_test = 0 (harus 0)
  [OK] overlap_review_id_val_test = 0 (harus 0)
  [OK] berkas split train terverifikasi
  [OK] berkas split val terverifikasi
  [OK] berkas split test terverifikasi
[OK] 5. Pembagian data (split 42): seluruh pemeriksaan umum lulus.


True

## 6. Ablasi proxy classifier P1 sampai P4, kalibrasi, dan diagnostik

**Tujuan.** Menjalankan empat tahap proxy pada skema stratified K-Fold yang identik (K = 5, shuffle, seed 42): P1 (CLS beku dan Logistic Regression), P2 (mean-pooling beku dan Logistic Regression), P3 (IndoBERT fine-tuned, Cross-Entropy) dan P4 (IndoBERT fine-tuned, CORN, proxy final). P3 dan P4 dikalibrasi dengan temperature scaling per fold. Untuk tiap proxy dihitung kualitas (agregat dan per kelas rating asli), kalibrasi, dan jumlah baris ter-flag.

**Rujukan proposal.** Subbab 3.5 dan 3.6; kalibrasi pada Subbab 2.1.8.

**Masukan.** runs/split_seed42/data/split_train.csv.

**Keluaran.** runs/split_seed42/ablation/ (P1 sampai P3 dan tabel ablasi), runs/split_seed42/diagnostics/ (P4), runs/split_seed42/proxy/ (berkas OOF dan berkas per fold).

**Perkiraan waktu GPU.** P1 dan P2 di bawah 15 menit; P3 sekitar 50 menit; P4 sekitar 50 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Perbandingan P1 sampai P4 menjadi jalur utama pemeriksaan sirkularitas. ECE kumulatif dihitung sebagai ECE biner per ambang pada kejadian {y > k}, dengan 10 bin lebar sama, dirata-rata atas K-1 = 4 ambang, terhadap label teramati. Monotonisitas diperiksa pada kumulatif langsung dari rantai CORN (setelah temperature scaling, sebelum konversi ke probabilitas kelas). Karena kumulatif rantai adalah hasil kali bilangan pada [0, 1], pemeriksaan ini adalah uji kewarasan numerik; bagian yang informatif adalah selisih antara kumulatif rantai dan kumulatif turunan (akibat pembatasan minimal 1e-8 dan normalisasi ulang) serta nilai ECE.

**Peringatan.** P1 sampai P3 tidak mengekspor sampel validasi manusia maupun varian data bersih. Hanya P4 yang melakukannya (bagian 7). Berkas per fold memungkinkan melanjutkan bila runtime putus.

In [14]:
df_train_main = pd.read_csv(ctx_main.split_file("train"))
ev_p1 = ablation.run_proxy_evaluation(ctx_main, df_train_main, 0, config.PROXY_CV_FOLDS, ctx_main.ablation_dir)
ev_p2 = ablation.run_proxy_evaluation(ctx_main, df_train_main, 1, config.PROXY_CV_FOLDS, ctx_main.ablation_dir)


=== P1 frozen_cls_lr (K=5) ===


D:\SKRIPSI\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Besto\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8130.95it/s]
                                                                  

   confident_learning: 4570 baris (45.93%), selisih terhadap estimasi: 1181  <-- DIPILIH
   prune_by_noise_rate: 3946 baris (39.66%), selisih terhadap estimasi: 1805
   estimasi teoretis (off_diagonal_calibrated): 5751 baris
Kualitas proxy: acc=0.4289 MAE=0.9484 off-by-1=0.7442 QWK=0.6100

=== P2 frozen_meanpool_lr (K=5) ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8728.99it/s]
                                                                   

   confident_learning: 4598 baris (46.22%), selisih terhadap estimasi: 1125  <-- DIPILIH
   prune_by_noise_rate: 3952 baris (39.72%), selisih terhadap estimasi: 1771
   estimasi teoretis (off_diagonal_calibrated): 5723 baris
Kualitas proxy: acc=0.4296 MAE=0.9599 off-by-1=0.7400 QWK=0.5990


In [15]:
ev_p3 = ablation.run_proxy_evaluation(ctx_main, df_train_main, 2, config.PROXY_CV_FOLDS, ctx_main.ablation_dir)


=== P3 finetuned_ce (K=5) ===
   [finetuned_ce] K=5 fold 1/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12039.39it/s]


      Epoch 1/3 - Loss: 1.3203
      Epoch 2/3 - Loss: 1.1586
      Epoch 3/3 - Loss: 0.9795
      Temperature scaling: T=1.2406 (NLL 1.3355 -> 1.2872)
   [finetuned_ce] K=5 fold 2/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7677.71it/s]


      Epoch 1/3 - Loss: 1.3093
      Epoch 2/3 - Loss: 1.1553
      Epoch 3/3 - Loss: 0.9705
      Temperature scaling: T=1.2424 (NLL 1.3266 -> 1.2762)
   [finetuned_ce] K=5 fold 3/5 (train=7959, val=1990)...


...

      Epoch 1/3 - Loss: 1.3200
      Epoch 2/3 - Loss: 1.1486
      Epoch 3/3 - Loss: 0.9760
      Temperature scaling: T=1.2730 (NLL 1.3527 -> 1.2815)
   [finetuned_ce] K=5 fold 4/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5038.25it/s]


      Epoch 1/3 - Loss: 1.3189
      Epoch 2/3 - Loss: 1.1670
      Epoch 3/3 - Loss: 0.9918
      Temperature scaling: T=1.1865 (NLL 1.2579 -> 1.2323)
   [finetuned_ce] K=5 fold 5/5 (train=7960, val=1989)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5286.04it/s]


      Epoch 1/3 - Loss: 1.3283
      Epoch 2/3 - Loss: 1.1538
      Epoch 3/3 - Loss: 0.9596
      Temperature scaling: T=1.2681 (NLL 1.3516 -> 1.2851)
   Temperature per fold: ['1.241', '1.242', '1.273', '1.186', '1.268'] (rata-rata 1.242)
   confident_learning: 4890 baris (49.15%), selisih terhadap estimasi: 665  <-- DIPILIH
   prune_by_noise_rate: 4372 baris (43.94%), selisih terhadap estimasi: 1183
   estimasi teoretis (off_diagonal_calibrated): 5555 baris
Kualitas proxy: acc=0.4509 MAE=0.7933 off-by-1=0.8190 QWK=0.6677


In [16]:
ev_p4 = ablation.run_proxy_evaluation(ctx_main, df_train_main, 3, config.PROXY_CV_FOLDS, ctx_main.diagnostics_dir)


=== P4 finetuned_corn (K=5) ===
   [finetuned_corn] K=5 fold 1/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8523.18it/s]


      Epoch 1/3 - Loss: 0.5074
      Epoch 2/3 - Loss: 0.4433
      Epoch 3/3 - Loss: 0.3766
      Temperature scaling: T=1.2084 (NLL 1.3513 -> 1.3103)
   [finetuned_corn] K=5 fold 2/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8269.76it/s]


      Epoch 1/3 - Loss: 0.5050
      Epoch 2/3 - Loss: 0.4422
      Epoch 3/3 - Loss: 0.3763
      Temperature scaling: T=1.1539 (NLL 1.2620 -> 1.2427)
   [finetuned_corn] K=5 fold 3/5 (train=7959, val=1990)...


...

      Epoch 1/3 - Loss: 0.5065
      Epoch 2/3 - Loss: 0.4423
      Epoch 3/3 - Loss: 0.3773
      Temperature scaling: T=1.2097 (NLL 1.3402 -> 1.2992)
   [finetuned_corn] K=5 fold 4/5 (train=7959, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5357.29it/s]


      Epoch 1/3 - Loss: 0.5131
      Epoch 2/3 - Loss: 0.4476
      Epoch 3/3 - Loss: 0.3840
      Temperature scaling: T=1.1174 (NLL 1.2627 -> 1.2522)
   [finetuned_corn] K=5 fold 5/5 (train=7960, val=1989)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7370.25it/s]


      Epoch 1/3 - Loss: 0.5074
      Epoch 2/3 - Loss: 0.4442
      Epoch 3/3 - Loss: 0.3767
      Temperature scaling: T=1.1485 (NLL 1.2824 -> 1.2631)
   Temperature per fold: ['1.208', '1.154', '1.210', '1.117', '1.148'] (rata-rata 1.168)
   confident_learning: 4757 baris (47.81%), selisih terhadap estimasi: 723  <-- DIPILIH
   prune_by_noise_rate: 4250 baris (42.72%), selisih terhadap estimasi: 1230
   estimasi teoretis (off_diagonal_calibrated): 5480 baris
Kualitas proxy: acc=0.4593 MAE=0.7900 off-by-1=0.8164 QWK=0.6754


### 6.4 Tabel ablasi (Tabel 3.1) dan diagnostik P4

In [17]:
ablation_table = pipeline.step_ablation_p1_p3(ctx_main, df_train_main, ev_p4)
display(ablation_table)


=== P1 frozen_cls_lr (K=5) ===
   confident_learning: 4570 baris (45.93%), selisih terhadap estimasi: 1181  <-- DIPILIH
   prune_by_noise_rate: 3946 baris (39.66%), selisih terhadap estimasi: 1805
   estimasi teoretis (off_diagonal_calibrated): 5751 baris
Kualitas proxy: acc=0.4289 MAE=0.9484 off-by-1=0.7442 QWK=0.6100

=== P2 frozen_meanpool_lr (K=5) ===
   confident_learning: 4598 baris (46.22%), selisih terhadap estimasi: 1125  <-- DIPILIH
   prune_by_noise_rate: 3952 baris (39.72%), selisih terhadap estimasi: 1771
   estimasi teoretis (off_diagonal_calibrated): 5723 baris
Kualitas proxy: acc=0.4296 MAE=0.9599 off-by-1=0.7400 QWK=0.5990

=== P3 finetuned_ce (K=5) ===
[OK] OOF finetuned_ce K=5 sudah ada, dimuat.
   confident_learning: 4890 baris (49.15%), selisih terhadap estimasi: 665  <-- DIPILIH
   prune_by_noise_rate: 4372 baris (43.94%), selisih terhadap estimasi: 1183
   estimasi teoretis (off_diagonal_calibrated): 5555 baris
Kualitas proxy: acc=0.4509 MAE=0.7933 off-by-1=0.81

,proxy_id,label,name,desc,accuracy,mae,rmse,off_by_one,qwk,pct_flagged_noise,selected_filter_method,n_issues_confident_learning,n_issues_prune_by_noise_rate,n_issues_estimated_theoretical,temperature_rata,ece_top_label,ece_kumulatif_turunan_rata,ece_kumulatif_rantai_rata
0,0,P1,frozen_cls_lr,Embedding beku token [CLS] + Logistic Regressi...,0.428887,0.948437,1.414356,0.744195,0.609989,45.934265,confident_learning,4570,3946,5751,NaN,0.061261,0.075827,NaN
1,1,P2,frozen_meanpool_lr,Embedding beku mean-pooling + Logistic Regress...,0.429591,0.959895,1.435027,0.739974,0.599002,46.215700,confident_learning,4598,3952,5723,NaN,0.067247,0.079229,NaN
2,2,P3,finetuned_ce,"IndoBERT fine-tuned K-Fold, Cross-Entropy, tem...",0.450900,0.793346,1.197251,0.818977,0.667714,49.150668,confident_learning,4890,4372,5555,1.242123,0.084656,0.036781,NaN
3,3,P4,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN, temperature ...",0.459343,0.790029,1.203322,0.816363,0.675357,47.813851,confident_learning,4757,4250,5480,1.167567,0.079641,0.042001,0.042001


In [18]:
res = ev_p4["result"]
cal_keys = [k for k in res if k.startswith("ece_") or k.startswith("monotonisitas_") or k.startswith("selisih_rantai")]
display(pd.Series({k: res[k] for k in cal_keys}, name="kalibrasi P4"))
display(pd.read_csv(os.path.join(ctx_main.diagnostics_dir, "P4__finetuned_corn__K5__temperature_per_fold.csv")))
display(ev_p4["per_class"])

ece_top_label                               7.964117e-02
ece_kumulatif_turunan_rata                  4.200110e-02
monotonisitas_turunan__n_baris              9.949000e+03
monotonisitas_turunan__n_baris_melanggar    0.000000e+00
monotonisitas_turunan__proporsi_monoton     1.000000e+00
monotonisitas_turunan__pelanggaran_maks    -1.980388e-04
ece_kumulatif_turunan_ambang1               5.184380e-02
ece_kumulatif_turunan_ambang2               5.590971e-02
ece_kumulatif_turunan_ambang3               3.774854e-02
ece_kumulatif_turunan_ambang4               2.250234e-02
ece_kumulatif_rantai_rata                   4.200110e-02
ece_kumulatif_rantai_ambang1                5.184380e-02
ece_kumulatif_rantai_ambang2                5.590971e-02
ece_kumulatif_rantai_ambang3                3.774854e-02
ece_kumulatif_rantai_ambang4                2.250234e-02
monotonisitas_rantai__n_baris               9.949000e+03
monotonisitas_rantai__n_baris_melanggar     0.000000e+00
monotonisitas_rantai__proporsi_

,fold,temperature
0,1,1.208372
1,2,1.153856
2,3,1.209693
3,4,1.117441
4,5,1.148474


,rating,n_label_asli,n_prediksi,recall,precision,mae
0,1,2605,3308,0.679846,0.535369,0.499040
1,2,2004,2037,0.310379,0.305351,0.815868
2,3,1866,1411,0.232047,0.306875,1.080922
3,4,1680,1197,0.282738,0.396825,1.077976
4,5,1794,1996,0.707358,0.635772,0.611483


In [19]:
selfcheck.check_ablation(ctx_main)

Pemeriksaan mandiri: 6. Ablasi proxy P1 sampai P4
  [OK] hasil P1 ada
  [OK] hasil P2 ada
  [OK] hasil P3 ada
  [OK] hasil P4 ada
  [OK] tabel ablasi ada
  [OK] varian data bersih hanya berasal dari P4
[OK] 6. Ablasi proxy P1 sampai P4: seluruh pemeriksaan umum lulus.


True

In [20]:
# Unduhan cadangan lokal (boleh dilewati)
bk = drive_io.export_backup_zip(config.DRIVE_ROOT, f"cadangan_bagian6_{RUN_TAG}.zip")
print(bk)
drive_io.download_if_colab(bk["zip_path"])

{'zip_path': 'D:\\SKRIPSI\\lokal\\cadangan_bagian6_20261003_224958.zip', 'n_files': 68, 'bytes': 7935091, 'sha256': '9249d3dc9124ed5c673bcc7be871ce312e60b00bd3490820222d2c405883e8c6'}
(Bukan Colab) berkas cadangan ada di: D:\SKRIPSI\lokal\cadangan_bagian6_20261003_224958.zip


## 7. Penetapan P4 sebagai proxy final, deteksi noise, pemilihan metode filter, resolusi konflik, dan tiga varian data

**Tujuan.** Menetapkan P4 sebagai proxy final (ditetapkan di muka), menjalankan Confident learning (min_examples_per_class = 20) dengan dua metode filter, memilih metode yang paling dekat dengan estimasi num_label_issues() varian off_diagonal_calibrated, menjalankan resolusi konflik teks identik pasca deteksi, dan menurunkan tiga varian data latih: raw (resolved), hard-pruned, dan severity-aware (jarak ordinal 2 atau lebih).

**Rujukan proposal.** Subbab 3.5 (aturan keputusan P4), 3.6, dan 3.8.

**Masukan.** OOF P4 dari bagian 6 (dimuat dari berkas, tidak dilatih ulang).

**Keluaran.** runs/split_seed42/cleaned/train_{raw,hard,severe}.csv; runs/split_seed42/noise/ (flags, baris ter-flag, ringkasan varian); runs/split_seed42/diagnostics/P4__jarak_ordinal_noise_per_kelas.csv.

**Perkiraan waktu GPU.** kurang dari 5 menit (tanpa pelatihan ulang) (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Tabel jarak ordinal per kelas memuat persentase baris ter-flag dengan jarak di atas ambang. Kolom penanda_confound bernilai YA bila kelas itu berada di dua teratas pada persentase jarak berat sekaligus di dua terbawah pada recall (atau precision) proxy, sehingga terindikasi confound antara proxy lemah dan kelas yang memang banyak berlabel salah.

**Peringatan.** Setelah penghapusan duplikat murni pada praproses, setiap grup teks konflik memuat satu baris per rating, sehingga seluruh grup konflik seri dan dibuang seluruhnya. Voting mayoritas karena itu tidak pernah memilih rating mayoritas. Hal ini dinyatakan apa adanya (lihat dokumen audit).

In [21]:
out_p4 = pipeline.step_p4(ctx_main, df_train_main)
display(pd.Series(out_p4["variant_stats"]))
display(out_p4["noise_by_class"])
df_noise_main = out_p4["df_noise"]


=== P4 finetuned_corn (K=5) ===
[OK] OOF finetuned_corn K=5 sudah ada, dimuat.
   confident_learning: 4757 baris (47.81%), selisih terhadap estimasi: 723  <-- DIPILIH
   prune_by_noise_rate: 4250 baris (42.72%), selisih terhadap estimasi: 1230
   estimasi teoretis (off_diagonal_calibrated): 5480 baris
Kualitas proxy: acc=0.4593 MAE=0.7900 off-by-1=0.8164 QWK=0.6754
Distribusi rating_diff pada baris noise (sebelum resolusi):
rating_diff
1    3126
2    1199
3     332
4     100
Baris dibuang (rating minoritas dalam grup konflik): 0
Baris dibuang (seri, tidak ada mayoritas jelas)    : 263
Total dibuang                                      : 263
Resolusi konflik teks identik (pasca deteksi): 9949 -> 9686 baris
Raw 9686 | Hard-prune: buang 4606, sisa 5080 | Severity-aware: buang 1554, sisa 8132


split_seed                                       42
n_latih_awal                                   9949
n_setelah_resolusi_konflik                     9686
n_dibuang_resolusi_konflik                      263
metode_filter_terpilih           confident_learning
n_flag_noise_sebelum_resolusi                  4757
n_raw                                          9686
n_hard                                         5080
n_severe                                       8132
n_dibuang_hard                                 4606
n_dibuang_severe                               1554
n_confident_learning                           4757
n_prune_by_noise_rate                          4250
n_estimated_theoretical                        5480
dtype: object

,rating,n_baris_kelas,n_flag,pct_flag_dalam_kelas,n_jarak1,n_jarak2,n_jarak3,n_jarak4,n_jarak_ge_ambang,pct_jarak_ge_ambang_dari_flag,recall_proxy,precision_proxy,mae_proxy,penanda_confound_recall,penanda_confound_precision
0,1,2605,816,31.324376,506,218,64,28,310,37.990196,0.679846,0.535369,0.499040,tidak,tidak
1,2,2004,1077,53.742515,892,132,53,0,185,17.177344,0.310379,0.305351,0.815868,tidak,tidak
2,3,1866,1316,70.525188,836,480,0,0,480,36.474164,0.232047,0.306875,1.080922,tidak,tidak
3,4,1680,1056,62.857143,662,258,136,0,394,37.310606,0.282738,0.396825,1.077976,tidak,tidak
4,5,1794,492,27.424749,230,111,79,72,262,53.252033,0.707358,0.635772,0.611483,tidak,tidak


In [22]:
selfcheck.check_p4(ctx_main)

Pemeriksaan mandiri: 7. P4, deteksi noise, dan varian (split 42)
  [OK] OOF P4 K=5 tersimpan
  [OK] bentuk OOF (9949, 5)
  [OK] probabilitas kelas berjumlah satu
  [OK] kumulatif rantai CORN tersimpan
  [OK] satu temperature per fold
  [OK] varian data latih 'raw' ada
  [OK] varian data latih 'hard' ada
  [OK] varian data latih 'severe' ada
  [OK] urutan ukuran hard <= severe <= raw ({'raw': 9686, 'hard': 5080, 'severe': 8132})
[OK] 7. P4, deteksi noise, dan varian (split 42): seluruh pemeriksaan umum lulus.


True

## 8. Validasi manusia

**Tujuan.** Mengukur kesepakatan antara Confident learning dan penilai manusia pada sampel baris ter-flag P4. Sampel dinilai buta (rating_diff dan prediksi tidak diperlihatkan), dengan tiga kategori: noise, not_noise, ambiguous.

**Rujukan proposal.** Subbab 3.7.

**Masukan.** Baris ter-flag P4 pada data latih split utama.

**Keluaran.** annotations/human_validation_sample.csv (+ sidecar hash), annotations/human_validation_sample_annotator2.csv (+ sidecar), runs/split_seed42/validation/human_validation_{result,per_bin,summary,kappa}.csv.

**Perkiraan waktu GPU.** tanpa GPU; waktu anotasi bergantung pada penilai (dapat dikerjakan paralel dengan pelatihan) (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Kesepakatan dilaporkan keseluruhan dan per bin jarak ordinal, dibahas deskriptif-komparatif tanpa ambang lulus atau gagal. Cohen's kappa biasa dipakai karena tiga kategori tidak berjenjang. Bin jarak 3 dan 4 berukuran kecil secara struktural (Subbab 3.8).

**Peringatan.** Dua alur. ALUR A: anotasi dari eksekusi sebelumnya dipakai kembali. Salin berkas sampel dan sidecar ke folder annotations, lalu sel verifikasi memeriksa hash daftar review_id ter-flag P4. Pada run murni, himpunan ter-flag dapat berbeda karena nondeterminisme GPU; bila hash tidak cocok, eksekusi berhenti dan anotasi harus dibuat ulang. ALUR B: sampel kosong dibuat, peneliti mengisi kolom human_verdict di Drive, lalu perhitungan dijalankan. Berkas anotasi tidak pernah ditimpa. Pelatihan (bagian 9) tidak perlu menunggu bagian ini selesai.

### 8.1 Ukuran sampel (Cochran) dan alokasi

In [23]:
df_noise_main = pd.read_csv(os.path.join(ctx_main.noise_dir, "flagged_rows_P4_K5.csv"))
cochran_hv = human_validation.report_sample_size(df_noise_main)

N baris ter-flag P4 (data latih)     : 4757
n0 = Z^2 p (1 - p) / e^2             : 96.04
n setelah koreksi populasi terbatas  : 95
n perencanaan yang dipakai (Subbab 3.7): 100


### 8.2 Alur A (verifikasi hash) atau alur B (membuat sampel baru)

In [24]:
if ALUR_ANOTASI_VALIDASI == "A":
    human_validation.verify_annotation(df_noise_main)
elif ALUR_ANOTASI_VALIDASI == "B":
    human_validation.create_sample(ctx_main, df_noise_main)
    human_validation.create_second_annotator()
else:
    raise ValueError("ALUR_ANOTASI_VALIDASI harus 'A' atau 'B'")

N baris ter-flag P4 (data latih)     : 4757
n0 = Z^2 p (1 - p) / e^2             : 96.04
n setelah koreksi populasi terbatas  : 95
n perencanaan yang dipakai (Subbab 3.7): 100
Alokasi per bin jarak ordinal: {2: 15, 3: 15, 4: 15, 1: 55} (total 100)
[OK] Sampel buta dibuat: D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\annotations\human_validation_sample.csv
Isi kolom 'human_verdict' dengan: noise / not_noise / ambiguous, lalu simpan di Drive.
[OK] Sampel penilai kedua: 30 baris -> D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\annotations\human_validation_sample_annotator2.csv


### 8.3 Hitung kesepakatan dan kappa

Jalankan sel ini setelah kolom human_verdict terisi lengkap. Boleh dijalankan setelah pelatihan selesai.
Penilai kedua bersifat opsional: bila tidak ada, gunakan jalur test-retest pada sub-bagian berikutnya.

In [25]:
summary_hv, breakdown_hv = human_validation.compute_agreement(ctx_main, df_noise_main)
display(summary_hv); display(breakdown_hv)

[OK] Hash sidecar validasi manusia cocok (n dasar = 4757).
[OK] Berkas penilai kedua konsisten dengan sampel.
Total sampel: 100 | setuju (noise): 15 (15.0%) | ambigu: 6 (6.0%) | tidak setuju (not_noise): 79 (79.0%)
 rating_diff  n_sampel  pct_noise  pct_not_noise  pct_ambiguous
           1        55        7.3           87.3            5.5
           2        15       13.3           80.0            6.7
           3        15       20.0           73.3            6.7
           4        15       40.0           53.3            6.7
Acuan deskriptif (Northcutt dkk., 2021, domain gambar): sekitar 58 persen sampel terbukti isu. Dibahas deskriptif-komparatif, tanpa ambang lulus atau gagal.


,n,noise,ambiguous,not_noise,agreement_rate
0,100,15,6,79,0.15


,rating_diff,n_sampel,pct_noise,pct_not_noise,pct_ambiguous
0,1,55,7.3,87.3,5.5
1,2,15,13.3,80.0,6.7
2,3,15,20.0,73.3,6.7
3,4,15,40.0,53.3,6.7


In [26]:
try:
    kappa_hv = human_validation.compute_interannotator_kappa(ctx_main)
except (FileNotFoundError, ValueError) as e:
    print("[PERINGATAN] Kappa antar-penilai belum dapat dihitung:", e)
    print("Bila hanya ada satu penilai, gunakan human_validation.create_retest_subset() lalu compute_test_retest(ctx_main).")

Cohen's kappa antar-penilai (n=30): 0.2603


In [27]:
selfcheck.check_annotation(ctx_main)

Pemeriksaan mandiri: 8. Anotasi manual
  [OK] human_validation: sidecar hash ada
  [PERINGATAN] gold: berkas anotasi belum ada (alur B belum dijalankan atau alur A belum disalin).
[OK] 8. Anotasi manual: seluruh pemeriksaan umum lulus.


True

## 9. Pelatihan enam skenario dan tiga seed

**Tujuan.** Melatih M1 sampai M6 (tiga varian data dikali dua fungsi loss) dengan tiga seed bobot (42, 123, 2024) pada konfigurasi Tabel 3.3: indobenchmark/indobert-base-p1, token maksimum 128, batch 16, learning rate 2e-5, AdamW weight decay 0,01, warmup linear 10 persen, maksimum 10 epoch, early stopping patience 3 (MAE validasi), mixed precision, checkpoint terbaik menurut QWK validasi (tie-breaker MAE), dropout 0,3, pooler_output.

**Rujukan proposal.** Subbab 3.9 (Tabel 3.2 dan 3.3).

**Masukan.** runs/split_seed42/cleaned/train_{raw,hard,severe}.csv, split_val.csv, split_test.csv.

**Keluaran.** Per unit (skenario, seed): runs/split_seed42/predictions/{skenario}__seed{seed}.csv (review_id, y_true, y_pred pada data uji) dan runs/split_seed42/training/{skenario}__seed{seed}.json. Checkpoint ditulis ke disk lokal Colab.

**Perkiraan waktu GPU.** 18 sesi pelatihan, sekitar 8 sampai 12 jam (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Setiap sel di bawah menjalankan satu skenario untuk tiga seed. Unit yang sudah terverifikasi dilewati otomatis.

**Peringatan.** Sel panjang. Bila runtime putus, jalankan ulang sel konfigurasi (LANJUTKAN_EKSEKUSI = True) dan sel yang terhenti. Berkas prediksi dan metrik ditulis sekali per unit dan tidak pernah ditimpa. Kriteria seed bobot terpisah dari seed partisi.

In [28]:
mode_main = pipeline.register_run_mode(ctx_main, SEEDS_MAIN, hemat=False)
print(mode_main)

{'split_seed': 42, 'weight_seeds': [42, 123, 2024], 'hemat': False, 'keterangan': 'konfigurasi penuh: tiga seed bobot'}


In [29]:
status = train.run_scenario(ctx_main, "M1_Baseline_CE", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M1_Baseline_CE"])


[MULAI] M1_Baseline_CE | split 42 | seed bobot 42 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9130.72it/s]


Epoch 1/10 - Loss: 1.3599 - Val MAE: 0.8263 | QWK: 0.6780 | Off-by-1: 0.7959 | Acc: 0.4499
Epoch 2/10 - Loss: 1.1871 - Val MAE: 0.7691 | QWK: 0.6940 | Off-by-1: 0.8256 | Acc: 0.4682
Epoch 3/10 - Loss: 1.0133 - Val MAE: 0.8016 | QWK: 0.6696 | Off-by-1: 0.8199 | Acc: 0.4492
Epoch 4/10 - Loss: 0.7812 - Val MAE: 0.8093 | QWK: 0.6716 | Off-by-1: 0.8171 | Acc: 0.4273
Epoch 5/10 - Loss: 0.5076 - Val MAE: 0.7712 | QWK: 0.6823 | Off-by-1: 0.8496 | Acc: 0.4209
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.6940): MAE=0.7440 | RMSE=1.1509 | Acc=0.4751 | Off-by-1=0.8389 | QWK=0.7111

[MULAI] M1_Baseline_CE | split 42 | seed bobot 123 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6378.51it/s]


Epoch 1/10 - Loss: 1.3661 - Val MAE: 0.8460 | QWK: 0.6756 | Off-by-1: 0.7825 | Acc: 0.4499
Epoch 2/10 - Loss: 1.1915 - Val MAE: 0.8270 | QWK: 0.6660 | Off-by-1: 0.7888 | Acc: 0.4640
Epoch 3/10 - Loss: 1.0110 - Val MAE: 0.7938 | QWK: 0.6617 | Off-by-1: 0.8157 | Acc: 0.4315
Epoch 4/10 - Loss: 0.7543 - Val MAE: 0.7881 | QWK: 0.6835 | Off-by-1: 0.8263 | Acc: 0.4329
Epoch 5/10 - Loss: 0.4722 - Val MAE: 0.8100 | QWK: 0.6627 | Off-by-1: 0.8157 | Acc: 0.4266
Epoch 6/10 - Loss: 0.2556 - Val MAE: 0.8347 | QWK: 0.6422 | Off-by-1: 0.8030 | Acc: 0.4110
Epoch 7/10 - Loss: 0.1460 - Val MAE: 0.8107 | QWK: 0.6603 | Off-by-1: 0.8150 | Acc: 0.4294
   Early stopping di epoch 7 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 4, QWK val 0.6835): MAE=0.7591 | RMSE=1.1580 | Acc=0.4643 | Off-by-1=0.8309 | QWK=0.6874

[MULAI] M1_Baseline_CE | split 42 | seed bobot 2024 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8515.70it/s]


Epoch 1/10 - Loss: 1.3536 - Val MAE: 0.8326 | QWK: 0.6422 | Off-by-1: 0.8213 | Acc: 0.3983
Epoch 2/10 - Loss: 1.1843 - Val MAE: 0.8114 | QWK: 0.6917 | Off-by-1: 0.7966 | Acc: 0.4605
Epoch 3/10 - Loss: 1.0196 - Val MAE: 0.7719 | QWK: 0.6965 | Off-by-1: 0.8220 | Acc: 0.4647
Epoch 4/10 - Loss: 0.7694 - Val MAE: 0.7895 | QWK: 0.6716 | Off-by-1: 0.8249 | Acc: 0.4336
Epoch 5/10 - Loss: 0.4859 - Val MAE: 0.8044 | QWK: 0.6581 | Off-by-1: 0.8234 | Acc: 0.4181
Epoch 6/10 - Loss: 0.2786 - Val MAE: 0.8100 | QWK: 0.6479 | Off-by-1: 0.8347 | Acc: 0.4082
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.6965): MAE=0.7626 | RMSE=1.1748 | Acc=0.4692 | Off-by-1=0.8323 | QWK=0.6977


,scenario,weight_seed,selesai
0,M1_Baseline_CE,42,True
1,M1_Baseline_CE,123,True
2,M1_Baseline_CE,2024,True


In [30]:
status = train.run_scenario(ctx_main, "M2_CleanedHard_CE", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M2_CleanedHard_CE"])


[MULAI] M2_CleanedHard_CE | split 42 | seed bobot 42 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7928.74it/s]


Epoch 1/10 - Loss: 1.0189 - Val MAE: 0.8030 | QWK: 0.6789 | Off-by-1: 0.8051 | Acc: 0.4513
Epoch 2/10 - Loss: 0.6128 - Val MAE: 0.7479 | QWK: 0.7073 | Off-by-1: 0.8475 | Acc: 0.4449
Epoch 3/10 - Loss: 0.3688 - Val MAE: 0.7556 | QWK: 0.7000 | Off-by-1: 0.8362 | Acc: 0.4534
Epoch 4/10 - Loss: 0.2105 - Val MAE: 0.7641 | QWK: 0.6972 | Off-by-1: 0.8185 | Acc: 0.4541
Epoch 5/10 - Loss: 0.1127 - Val MAE: 0.7634 | QWK: 0.6908 | Off-by-1: 0.8270 | Acc: 0.4534
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.7073): MAE=0.7335 | RMSE=1.1084 | Acc=0.4562 | Off-by-1=0.8589 | QWK=0.7109

[MULAI] M2_CleanedHard_CE | split 42 | seed bobot 123 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7891.41it/s]


Epoch 1/10 - Loss: 1.0466 - Val MAE: 0.8157 | QWK: 0.6676 | Off-by-1: 0.8001 | Acc: 0.4590
Epoch 2/10 - Loss: 0.6088 - Val MAE: 0.7853 | QWK: 0.6885 | Off-by-1: 0.8100 | Acc: 0.4696
Epoch 3/10 - Loss: 0.3703 - Val MAE: 0.7458 | QWK: 0.7104 | Off-by-1: 0.8411 | Acc: 0.4569
Epoch 4/10 - Loss: 0.2140 - Val MAE: 0.7627 | QWK: 0.6945 | Off-by-1: 0.8369 | Acc: 0.4470
Epoch 5/10 - Loss: 0.1125 - Val MAE: 0.7486 | QWK: 0.7061 | Off-by-1: 0.8383 | Acc: 0.4576
Epoch 6/10 - Loss: 0.0633 - Val MAE: 0.7267 | QWK: 0.7129 | Off-by-1: 0.8482 | Acc: 0.4605
Epoch 7/10 - Loss: 0.0317 - Val MAE: 0.7394 | QWK: 0.7093 | Off-by-1: 0.8404 | Acc: 0.4605
Epoch 8/10 - Loss: 0.0160 - Val MAE: 0.7345 | QWK: 0.7144 | Off-by-1: 0.8446 | Acc: 0.4640
Epoch 9/10 - Loss: 0.0085 - Val MAE: 0.7401 | QWK: 0.7117 | Off-by-1: 0.8411 | Acc: 0.4626
   Early stopping di epoch 9 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 8, QWK val 0.7144): MAE=0.7360 | RMSE=1.1305 | Acc=0.4695 | Off-by-1=0.8477 | QWK=0.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7428.83it/s]


Epoch 1/10 - Loss: 1.0204 - Val MAE: 0.8023 | QWK: 0.6749 | Off-by-1: 0.8107 | Acc: 0.4484
Epoch 2/10 - Loss: 0.6152 - Val MAE: 0.7691 | QWK: 0.6988 | Off-by-1: 0.8234 | Acc: 0.4612
Epoch 3/10 - Loss: 0.3758 - Val MAE: 0.7924 | QWK: 0.6842 | Off-by-1: 0.8129 | Acc: 0.4541
Epoch 4/10 - Loss: 0.2175 - Val MAE: 0.7401 | QWK: 0.7102 | Off-by-1: 0.8453 | Acc: 0.4541
Epoch 5/10 - Loss: 0.1182 - Val MAE: 0.7281 | QWK: 0.7178 | Off-by-1: 0.8453 | Acc: 0.4640
Epoch 6/10 - Loss: 0.0634 - Val MAE: 0.7514 | QWK: 0.7095 | Off-by-1: 0.8411 | Acc: 0.4548
Epoch 7/10 - Loss: 0.0372 - Val MAE: 0.7302 | QWK: 0.7123 | Off-by-1: 0.8362 | Acc: 0.4753
Epoch 8/10 - Loss: 0.0170 - Val MAE: 0.7253 | QWK: 0.7161 | Off-by-1: 0.8376 | Acc: 0.4767
Epoch 9/10 - Loss: 0.0133 - Val MAE: 0.7218 | QWK: 0.7199 | Off-by-1: 0.8411 | Acc: 0.4753
Epoch 10/10 - Loss: 0.0088 - Val MAE: 0.7175 | QWK: 0.7197 | Off-by-1: 0.8439 | Acc: 0.4760
[SELESAI] Uji (checkpoint epoch 9, QWK val 0.7199): MAE=0.7342 | RMSE=1.1144 | Acc=0.4636

,scenario,weight_seed,selesai
3,M2_CleanedHard_CE,42,True
4,M2_CleanedHard_CE,123,True
5,M2_CleanedHard_CE,2024,True


In [31]:
status = train.run_scenario(ctx_main, "M3_CleanedSevere_CE", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M3_CleanedSevere_CE"])


[MULAI] M3_CleanedSevere_CE | split 42 | seed bobot 42 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8019.86it/s]


Epoch 1/10 - Loss: 1.2014 - Val MAE: 0.7846 | QWK: 0.6834 | Off-by-1: 0.8107 | Acc: 0.4626
Epoch 2/10 - Loss: 0.9497 - Val MAE: 0.7493 | QWK: 0.6977 | Off-by-1: 0.8326 | Acc: 0.4633
Epoch 3/10 - Loss: 0.7688 - Val MAE: 0.7698 | QWK: 0.6909 | Off-by-1: 0.8220 | Acc: 0.4590
Epoch 4/10 - Loss: 0.5600 - Val MAE: 0.7592 | QWK: 0.6884 | Off-by-1: 0.8460 | Acc: 0.4386
Epoch 5/10 - Loss: 0.3568 - Val MAE: 0.7592 | QWK: 0.6938 | Off-by-1: 0.8475 | Acc: 0.4258
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.6977): MAE=0.7209 | RMSE=1.0996 | Acc=0.4709 | Off-by-1=0.8501 | QWK=0.7190

[MULAI] M3_CleanedSevere_CE | split 42 | seed bobot 123 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5869.01it/s]


Epoch 1/10 - Loss: 1.2259 - Val MAE: 0.7429 | QWK: 0.7009 | Off-by-1: 0.8347 | Acc: 0.4732
Epoch 2/10 - Loss: 0.9574 - Val MAE: 0.7387 | QWK: 0.7123 | Off-by-1: 0.8340 | Acc: 0.4739
Epoch 3/10 - Loss: 0.7622 - Val MAE: 0.7331 | QWK: 0.7071 | Off-by-1: 0.8460 | Acc: 0.4619
Epoch 4/10 - Loss: 0.5448 - Val MAE: 0.7239 | QWK: 0.7140 | Off-by-1: 0.8552 | Acc: 0.4527
Epoch 5/10 - Loss: 0.3444 - Val MAE: 0.7500 | QWK: 0.6963 | Off-by-1: 0.8376 | Acc: 0.4597
Epoch 6/10 - Loss: 0.2027 - Val MAE: 0.7606 | QWK: 0.6867 | Off-by-1: 0.8397 | Acc: 0.4435
Epoch 7/10 - Loss: 0.1153 - Val MAE: 0.7514 | QWK: 0.6880 | Off-by-1: 0.8496 | Acc: 0.4379
   Early stopping di epoch 7 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 4, QWK val 0.7140): MAE=0.7181 | RMSE=1.0845 | Acc=0.4618 | Off-by-1=0.8599 | QWK=0.7116

[MULAI] M3_CleanedSevere_CE | split 42 | seed bobot 2024 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6257.43it/s]


Epoch 1/10 - Loss: 1.2147 - Val MAE: 0.7606 | QWK: 0.6933 | Off-by-1: 0.8333 | Acc: 0.4421
Epoch 2/10 - Loss: 0.9540 - Val MAE: 0.7578 | QWK: 0.6979 | Off-by-1: 0.8390 | Acc: 0.4492
Epoch 3/10 - Loss: 0.7707 - Val MAE: 0.7394 | QWK: 0.7061 | Off-by-1: 0.8517 | Acc: 0.4548
Epoch 4/10 - Loss: 0.5536 - Val MAE: 0.7888 | QWK: 0.6739 | Off-by-1: 0.8326 | Acc: 0.4258
Epoch 5/10 - Loss: 0.3589 - Val MAE: 0.7451 | QWK: 0.7050 | Off-by-1: 0.8475 | Acc: 0.4499
Epoch 6/10 - Loss: 0.2158 - Val MAE: 0.7641 | QWK: 0.6927 | Off-by-1: 0.8425 | Acc: 0.4428
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7061): MAE=0.7276 | RMSE=1.1042 | Acc=0.4650 | Off-by-1=0.8508 | QWK=0.7136


,scenario,weight_seed,selesai
6,M3_CleanedSevere_CE,42,True
7,M3_CleanedSevere_CE,123,True
8,M3_CleanedSevere_CE,2024,True


In [32]:
status = train.run_scenario(ctx_main, "M4_Baseline_CORN", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M4_Baseline_CORN"])


[MULAI] M4_Baseline_CORN | split 42 | seed bobot 42 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8900.45it/s]


Epoch 1/10 - Loss: 0.5371 - Val MAE: 0.7535 | QWK: 0.6973 | Off-by-1: 0.8453 | Acc: 0.4379
Epoch 2/10 - Loss: 0.4611 - Val MAE: 0.7196 | QWK: 0.7194 | Off-by-1: 0.8609 | Acc: 0.4477
Epoch 3/10 - Loss: 0.3947 - Val MAE: 0.7451 | QWK: 0.6979 | Off-by-1: 0.8559 | Acc: 0.4315
Epoch 4/10 - Loss: 0.3057 - Val MAE: 0.7726 | QWK: 0.6703 | Off-by-1: 0.8566 | Acc: 0.4096
Epoch 5/10 - Loss: 0.2062 - Val MAE: 0.7761 | QWK: 0.6730 | Off-by-1: 0.8397 | Acc: 0.4181
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.7194): MAE=0.7136 | RMSE=1.0628 | Acc=0.4513 | Off-by-1=0.8697 | QWK=0.7172

[MULAI] M4_Baseline_CORN | split 42 | seed bobot 123 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8444.54it/s]


Epoch 1/10 - Loss: 0.5395 - Val MAE: 0.7556 | QWK: 0.6915 | Off-by-1: 0.8559 | Acc: 0.4216
Epoch 2/10 - Loss: 0.4634 - Val MAE: 0.7316 | QWK: 0.7144 | Off-by-1: 0.8595 | Acc: 0.4484
Epoch 3/10 - Loss: 0.3969 - Val MAE: 0.7514 | QWK: 0.6949 | Off-by-1: 0.8446 | Acc: 0.4456
Epoch 4/10 - Loss: 0.3113 - Val MAE: 0.7458 | QWK: 0.6960 | Off-by-1: 0.8439 | Acc: 0.4456
Epoch 5/10 - Loss: 0.2195 - Val MAE: 0.7472 | QWK: 0.7058 | Off-by-1: 0.8468 | Acc: 0.4499
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.7144): MAE=0.7097 | RMSE=1.0735 | Acc=0.4615 | Off-by-1=0.8697 | QWK=0.7250

[MULAI] M4_Baseline_CORN | split 42 | seed bobot 2024 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6537.64it/s]


Epoch 1/10 - Loss: 0.5392 - Val MAE: 0.7444 | QWK: 0.6963 | Off-by-1: 0.8545 | Acc: 0.4308
Epoch 2/10 - Loss: 0.4650 - Val MAE: 0.7415 | QWK: 0.7047 | Off-by-1: 0.8453 | Acc: 0.4506
Epoch 3/10 - Loss: 0.3971 - Val MAE: 0.7316 | QWK: 0.7092 | Off-by-1: 0.8503 | Acc: 0.4555
Epoch 4/10 - Loss: 0.3044 - Val MAE: 0.7599 | QWK: 0.6906 | Off-by-1: 0.8510 | Acc: 0.4308
Epoch 5/10 - Loss: 0.2031 - Val MAE: 0.8072 | QWK: 0.6628 | Off-by-1: 0.8129 | Acc: 0.4336
Epoch 6/10 - Loss: 0.1274 - Val MAE: 0.8150 | QWK: 0.6580 | Off-by-1: 0.8164 | Acc: 0.4230
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7092): MAE=0.7118 | RMSE=1.0754 | Acc=0.4629 | Off-by-1=0.8641 | QWK=0.7201


,scenario,weight_seed,selesai
9,M4_Baseline_CORN,42,True
10,M4_Baseline_CORN,123,True
11,M4_Baseline_CORN,2024,True


In [33]:
status = train.run_scenario(ctx_main, "M5_CleanedHard_CORN", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M5_CleanedHard_CORN"])


[MULAI] M5_CleanedHard_CORN | split 42 | seed bobot 42 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9745.08it/s]


Epoch 1/10 - Loss: 0.4505 - Val MAE: 0.7691 | QWK: 0.7046 | Off-by-1: 0.8347 | Acc: 0.4421
Epoch 2/10 - Loss: 0.2542 - Val MAE: 0.7479 | QWK: 0.7064 | Off-by-1: 0.8397 | Acc: 0.4506
Epoch 3/10 - Loss: 0.1593 - Val MAE: 0.7451 | QWK: 0.7125 | Off-by-1: 0.8503 | Acc: 0.4435
Epoch 4/10 - Loss: 0.0946 - Val MAE: 0.7500 | QWK: 0.7063 | Off-by-1: 0.8404 | Acc: 0.4548
Epoch 5/10 - Loss: 0.0570 - Val MAE: 0.7535 | QWK: 0.6996 | Off-by-1: 0.8411 | Acc: 0.4527
Epoch 6/10 - Loss: 0.0353 - Val MAE: 0.7408 | QWK: 0.7083 | Off-by-1: 0.8397 | Acc: 0.4626
Epoch 7/10 - Loss: 0.0215 - Val MAE: 0.7479 | QWK: 0.7026 | Off-by-1: 0.8362 | Acc: 0.4605
Epoch 8/10 - Loss: 0.0123 - Val MAE: 0.7345 | QWK: 0.7098 | Off-by-1: 0.8432 | Acc: 0.4640
Epoch 9/10 - Loss: 0.0078 - Val MAE: 0.7345 | QWK: 0.7097 | Off-by-1: 0.8411 | Acc: 0.4654
Epoch 10/10 - Loss: 0.0066 - Val MAE: 0.7373 | QWK: 0.7081 | Off-by-1: 0.8411 | Acc: 0.4626
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7125): MAE=0.7290 | RMSE=1.1045 | Acc=0.4601

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10430.85it/s]


Epoch 1/10 - Loss: 0.4462 - Val MAE: 0.8651 | QWK: 0.6642 | Off-by-1: 0.7726 | Acc: 0.4407
Epoch 2/10 - Loss: 0.2567 - Val MAE: 0.7585 | QWK: 0.7049 | Off-by-1: 0.8355 | Acc: 0.4520
Epoch 3/10 - Loss: 0.1675 - Val MAE: 0.7585 | QWK: 0.6962 | Off-by-1: 0.8326 | Acc: 0.4492
Epoch 4/10 - Loss: 0.1023 - Val MAE: 0.7323 | QWK: 0.7160 | Off-by-1: 0.8446 | Acc: 0.4612
Epoch 5/10 - Loss: 0.0620 - Val MAE: 0.7295 | QWK: 0.7160 | Off-by-1: 0.8439 | Acc: 0.4661
Epoch 6/10 - Loss: 0.0300 - Val MAE: 0.7239 | QWK: 0.7149 | Off-by-1: 0.8503 | Acc: 0.4633
Epoch 7/10 - Loss: 0.0231 - Val MAE: 0.7288 | QWK: 0.7171 | Off-by-1: 0.8468 | Acc: 0.4612
Epoch 8/10 - Loss: 0.0144 - Val MAE: 0.7366 | QWK: 0.7133 | Off-by-1: 0.8369 | Acc: 0.4626
Epoch 9/10 - Loss: 0.0095 - Val MAE: 0.7408 | QWK: 0.7103 | Off-by-1: 0.8390 | Acc: 0.4597
   Early stopping di epoch 9 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 7, QWK val 0.7171): MAE=0.7209 | RMSE=1.0989 | Acc=0.4681 | Off-by-1=0.8550 | QWK=0.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8216.35it/s]


Epoch 1/10 - Loss: 0.4477 - Val MAE: 0.7691 | QWK: 0.6853 | Off-by-1: 0.8340 | Acc: 0.4386
Epoch 2/10 - Loss: 0.2587 - Val MAE: 0.7465 | QWK: 0.7039 | Off-by-1: 0.8411 | Acc: 0.4513
Epoch 3/10 - Loss: 0.1611 - Val MAE: 0.7267 | QWK: 0.7182 | Off-by-1: 0.8453 | Acc: 0.4661
Epoch 4/10 - Loss: 0.0974 - Val MAE: 0.7253 | QWK: 0.7162 | Off-by-1: 0.8524 | Acc: 0.4569
Epoch 5/10 - Loss: 0.0556 - Val MAE: 0.7316 | QWK: 0.7088 | Off-by-1: 0.8460 | Acc: 0.4562
Epoch 6/10 - Loss: 0.0322 - Val MAE: 0.7154 | QWK: 0.7196 | Off-by-1: 0.8538 | Acc: 0.4675
Epoch 7/10 - Loss: 0.0192 - Val MAE: 0.7196 | QWK: 0.7211 | Off-by-1: 0.8496 | Acc: 0.4703
Epoch 8/10 - Loss: 0.0122 - Val MAE: 0.7267 | QWK: 0.7196 | Off-by-1: 0.8453 | Acc: 0.4661
Epoch 9/10 - Loss: 0.0089 - Val MAE: 0.7182 | QWK: 0.7201 | Off-by-1: 0.8482 | Acc: 0.4718
   Early stopping di epoch 9 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 7, QWK val 0.7211): MAE=0.7230 | RMSE=1.1034 | Acc=0.4685 | Off-by-1=0.8543 | QWK=0.

,scenario,weight_seed,selesai
12,M5_CleanedHard_CORN,42,True
13,M5_CleanedHard_CORN,123,True
14,M5_CleanedHard_CORN,2024,True


In [4]:
status = train.run_scenario(ctx_main, "M6_CleanedSevere_CORN", SEEDS_MAIN, copy_ckpt_to_drive=SALIN_CHECKPOINT_KE_DRIVE)
display(status[status["scenario"] == "M6_CleanedSevere_CORN"])


[MULAI] M6_CleanedSevere_CORN | split 42 | seed bobot 42 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9024.69it/s]


Epoch 1/10 - Loss: 0.4925 - Val MAE: 0.7507 | QWK: 0.7051 | Off-by-1: 0.8425 | Acc: 0.4456
Epoch 2/10 - Loss: 0.3786 - Val MAE: 0.7140 | QWK: 0.7209 | Off-by-1: 0.8588 | Acc: 0.4654
Epoch 3/10 - Loss: 0.3031 - Val MAE: 0.7408 | QWK: 0.7048 | Off-by-1: 0.8439 | Acc: 0.4605
Epoch 4/10 - Loss: 0.2215 - Val MAE: 0.7373 | QWK: 0.7048 | Off-by-1: 0.8376 | Acc: 0.4654
Epoch 5/10 - Loss: 0.1549 - Val MAE: 0.7549 | QWK: 0.7006 | Off-by-1: 0.8453 | Acc: 0.4456
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.7209): MAE=0.6971 | RMSE=1.0613 | Acc=0.4709 | Off-by-1=0.8697 | QWK=0.7344

[MULAI] M6_CleanedSevere_CORN | split 42 | seed bobot 123 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7120.94it/s]


Epoch 1/10 - Loss: 0.4995 - Val MAE: 0.7338 | QWK: 0.7177 | Off-by-1: 0.8588 | Acc: 0.4421
Epoch 2/10 - Loss: 0.3813 - Val MAE: 0.7387 | QWK: 0.7049 | Off-by-1: 0.8545 | Acc: 0.4379
Epoch 3/10 - Loss: 0.3065 - Val MAE: 0.7154 | QWK: 0.7180 | Off-by-1: 0.8552 | Acc: 0.4605
Epoch 4/10 - Loss: 0.2285 - Val MAE: 0.7641 | QWK: 0.6824 | Off-by-1: 0.8482 | Acc: 0.4280
Epoch 5/10 - Loss: 0.1582 - Val MAE: 0.7465 | QWK: 0.7049 | Off-by-1: 0.8566 | Acc: 0.4343
Epoch 6/10 - Loss: 0.1012 - Val MAE: 0.7394 | QWK: 0.7017 | Off-by-1: 0.8559 | Acc: 0.4400
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7180): MAE=0.7041 | RMSE=1.0659 | Acc=0.4671 | Off-by-1=0.8652 | QWK=0.7229

[MULAI] M6_CleanedSevere_CORN | split 42 | seed bobot 2024 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7656.37it/s]


Epoch 1/10 - Loss: 0.4936 - Val MAE: 0.7528 | QWK: 0.7069 | Off-by-1: 0.8390 | Acc: 0.4534
Epoch 2/10 - Loss: 0.3791 - Val MAE: 0.7281 | QWK: 0.7069 | Off-by-1: 0.8573 | Acc: 0.4520
Epoch 3/10 - Loss: 0.3054 - Val MAE: 0.7267 | QWK: 0.7127 | Off-by-1: 0.8524 | Acc: 0.4605
Epoch 4/10 - Loss: 0.2263 - Val MAE: 0.7422 | QWK: 0.7045 | Off-by-1: 0.8453 | Acc: 0.4583
Epoch 5/10 - Loss: 0.1541 - Val MAE: 0.7549 | QWK: 0.6933 | Off-by-1: 0.8475 | Acc: 0.4350
Epoch 6/10 - Loss: 0.0985 - Val MAE: 0.7422 | QWK: 0.7007 | Off-by-1: 0.8482 | Acc: 0.4477
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7127): MAE=0.7195 | RMSE=1.0912 | Acc=0.4650 | Off-by-1=0.8564 | QWK=0.7190


,scenario,weight_seed,selesai
15,M6_CleanedSevere_CORN,42,True
16,M6_CleanedSevere_CORN,123,True
17,M6_CleanedSevere_CORN,2024,True


In [5]:
selfcheck.check_training(ctx_main)

Pemeriksaan mandiri: 9. Pelatihan (split 42)
  [OK] run_mode.json ada
  [OK] sesi pelatihan lengkap 18/18
[OK] 9. Pelatihan (split 42): seluruh pemeriksaan umum lulus.


True

In [10]:
import json, glob, os
tot = 0.0
for p in sorted(glob.glob(os.path.join(ctx_main.training_dir, "M*__seed*.json"))):
    tot += json.load(open(p, encoding="utf-8"))["detik"]
print("total jam 18 sesi:", round(tot / 3600, 2))

total jam 18 sesi: 3.82


## 10. Evaluasi dan uji signifikansi H1 sampai H5

**Tujuan.** Menghitung tabel hasil enam skenario (rata-rata dan simpangan baku tiga seed), uji Wilcoxon Signed-Rank dua arah atas rata-rata absolute error per sampel (agregasi tiga seed) dengan koreksi Holm-Bonferroni atas lima hipotesis, effect size MAE dengan CI bootstrap 95 persen, selisih QWK ensemble dengan CI bootstrap (definisi utama), dan selisih QWK rata-rata per seed dengan CI bootstrap (analisis sensitivitas, tambahan).

**Rujukan proposal.** Subbab 3.11 dan 3.12 (Tabel 3.4: H1 M4 vs M1, H2 M6 vs M4, H3 M5 vs M4, H4 M6 vs M5, H5 M3 vs M1).

**Masukan.** Berkas prediksi tersimpan (tanpa memuat ulang checkpoint).

**Keluaran.** runs/split_seed42/significance/{final_results_table, significance_test, effect_sizes_mae, qwk_ensemble_effect, qwk_per_seed_effect, tabel_panjang_efek}.csv.

**Perkiraan waktu GPU.** tanpa GPU, beberapa menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Arah: untuk MAE, selisih (A minus B) negatif berarti model A lebih baik. Untuk QWK, selisih positif berarti model A lebih baik. QWK ensemble: prediksi tiga seed dirata-rata lalu dibulatkan. QWK per seed: QWK dihitung per seed lalu dirata-rata. Selisih QWK antar fungsi loss dapat berbeda antara kedua cara agregasi. Bila Wilcoxon (setelah koreksi) dan CI bootstrap tidak sejalan, keduanya dilaporkan bersama.

**Peringatan.** QWK tidak dapat didekomposisi per sampel sehingga tidak diuji dengan Wilcoxon. Data uji berpotensi masih mengandung noise.

In [6]:
out_sig = pipeline.step_analysis(ctx_main, SEEDS_MAIN)
display(out_sig["final"]); display(out_sig["sig"]); display(out_sig["eff"])
display(out_sig["qwk_ens"]); display(out_sig["qwk_per"])

,Model,n_seed,MAE_mean,MAE_std,RMSE_mean,RMSE_std,Acc_mean,Acc_std,Off-by-1_mean,Off-by-1_std,QWK_mean,QWK_std
0,M6_CleanedSevere_CORN,3,0.706933,0.009360,1.072833,0.013158,0.467670,0.002465,0.863796,0.005521,0.725430,0.006530
1,M4_Baseline_CORN,3,0.711718,0.001575,1.070558,0.005541,0.458567,0.005149,0.867880,0.002641,0.720740,0.003239
2,M3_CleanedSevere_CE,3,0.722222,0.003965,1.096073,0.008403,0.465920,0.003775,0.853641,0.004466,0.714731,0.003117
3,M5_CleanedHard_CORN,3,0.724323,0.003411,1.102261,0.002403,0.465570,0.003881,0.855976,0.001838,0.716059,0.003105
4,M2_CleanedHard_CE,3,0.734594,0.001031,1.117786,0.009310,0.463119,0.005442,0.851424,0.005282,0.710596,0.002224
5,M1_Baseline_CE,3,0.755252,0.008051,1.161259,0.010035,0.469538,0.004438,0.834034,0.003513,0.698720,0.009702


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.711718,0.755252,6.037154e-07,0.000003,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.706933,0.711718,3.689877e-01,0.368988,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.724323,0.711718,7.463251e-02,0.149265,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.706933,0.724323,1.310063e-02,0.039302,Ya
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.722222,0.755252,3.021414e-04,0.001209,Ya


,hypothesis,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,-0.043534,-0.058476,-0.028942
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.004785,-0.017857,0.007703
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.012605,-0.001401,0.026964
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.017390,-0.030932,-0.004202
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,-0.033030,-0.047855,-0.018201


,hypothesis,model_a,model_b,qwk_a,qwk_b,qwk_diff,ci_95_low,ci_95_high
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.730484,0.724966,0.005518,-0.004627,0.015110
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.730364,0.730484,-0.000120,-0.008865,0.008112
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.722318,0.730484,-0.008166,-0.018161,0.001857
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.730364,0.722318,0.008046,-0.000908,0.016974
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.727413,0.724966,0.002447,-0.007614,0.012159


,hypothesis,model_a,model_b,n_seed,qwk_a_rata_seed,qwk_b_rata_seed,qwk_diff,ci_95_low,ci_95_high,qwk_diff_seed42,qwk_diff_seed123,qwk_diff_seed2024
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,3,0.720740,0.698720,0.022019,0.013518,0.030143,0.006067,0.037610,0.022381
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,3,0.725430,0.720740,0.004691,-0.002816,0.012131,0.017224,-0.002074,-0.001078
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,3,0.716059,0.720740,-0.004681,-0.012820,0.003553,-0.005352,-0.007753,-0.000939
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,3,0.725430,0.716059,0.009372,0.002206,0.016424,0.022576,0.005678,-0.000139
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,3,0.714731,0.698720,0.016011,0.007094,0.024446,0.007896,0.024209,0.015926


In [7]:
selfcheck.check_significance(ctx_main)

Pemeriksaan mandiri: 10. Uji signifikansi (split 42)
  [OK] ada: final_results_table.csv
  [OK] ada: significance_test.csv
  [OK] ada: effect_sizes_mae.csv
  [OK] ada: qwk_ensemble_effect.csv
  [OK] ada: qwk_per_seed_effect.csv
  [OK] ada: tabel_panjang_efek.csv
  [OK] jumlah hipotesis pada tabel = 5
[OK] 10. Uji signifikansi (split 42): seluruh pemeriksaan umum lulus.


True

In [8]:
bk = drive_io.export_backup_zip(config.DRIVE_ROOT, f"cadangan_bagian10_{RUN_TAG}.zip")
print(bk)
drive_io.download_if_colab(bk["zip_path"])

{'zip_path': 'D:\\SKRIPSI\\lokal\\cadangan_bagian10_20261004_060807.zip', 'n_files': 129, 'bytes': 11945017, 'sha256': 'c6aa66f2a76f2c1a60b203bcf4a89247cab7ce01888b1fa8ed09b30ae1196a82'}
(Bukan Colab) berkas cadangan ada di: D:\SKRIPSI\lokal\cadangan_bagian10_20261004_060807.zip


## 11. Subset uji emas

**Tujuan.** Memeriksa ketahanan hasil H1 sampai H5 pada subset data uji yang dilabeli ulang oleh manusia. Sampel diambil stratified berdasarkan rating asli dari data uji split utama, dengan ukuran sesuai rumus Cochran (e = 10 persen).

**Rujukan proposal.** Subbab 3.10.

**Masukan.** runs/split_seed42/data/split_test.csv dan berkas prediksi tersimpan.

**Keluaran.** annotations/gold_test_sample.csv (+ sidecar), annotations/gold_test_sample_annotator2.csv (+ sidecar), runs/split_seed42/gold/gold_test_{evaluation,effect_sizes,undetermined_summary,kappa}.csv.

**Perkiraan waktu GPU.** tanpa GPU; waktu anotasi bergantung pada penilai (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Opsi ND dikeluarkan dari metrik dan proporsinya dilaporkan terpisah. Hanya arah keunggulan dan CI bootstrap yang dilaporkan, tanpa Wilcoxon kedua. Label emas final adalah label penilai pertama. Penilai kedua dipakai untuk kappa berbobot kuadratik pada baris overlap.

**Peringatan.** Rubrik operasional per kelas rating tersedia pada config_files/rubrik_anotasi_rating.md. Klaim 'rubrik disediakan' pada laporan hanya boleh dipertahankan bila penilai memang memakainya. Sampel dapat dibuat segera setelah bagian 5 dan diisi paralel dengan pelatihan. Komponen ini ditolak untuk split selain split utama.

### 11.1 Alur A (verifikasi hash) atau alur B (membuat sampel baru)

In [9]:
if ALUR_ANOTASI_GOLD == "A":
    gold_test.verify_annotation(ctx_main)
elif ALUR_ANOTASI_GOLD == "B":
    gold_test.create_sample(ctx_main)
    gold_test.create_second_annotator()
else:
    raise ValueError("ALUR_ANOTASI_GOLD harus 'A' atau 'B'")
print(open(os.path.join(REPO_DIR, "config_files", "rubrik_anotasi_rating.md"), encoding="utf-8").read())

Populasi data uji N=2856 | n0=96.04 | n Cochran (e=10%)=93
[OK] Sampel uji emas (buta) dibuat: D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\annotations\gold_test_sample.csv
Isi 'human_gold_rating' dengan 1 sampai 5, atau 'ND' bila rating wajar bergantung pada informasi di luar teks. Rating asli dan prediksi model tidak ditampilkan.
[OK] Sampel penilai kedua uji emas: 28 baris -> D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\annotations\gold_test_sample_annotator2.csv
# Rubrik Operasional Anotasi Rating (Subbab 3.10)

Rubrik ini disediakan sebagai acuan konsistensi penilai uji emas. Penilai membaca teks hasil
praproses (case folding, normalisasi slang, dan normalisasi emoji), bukan teks asli hasil scraping.
Rating asli dan prediksi model apa pun tidak diperlihatkan.

Catatan penggunaan: klaim "rubrik operasional disediakan" pada laporan hanya boleh dipertahankan
bila penilai memang memakai rubrik ini saat menganotasi. Bila tidak dipakai, nyatakan apa adanya.

Prinsip umum: nilai rating yang paling wajar diberik

### 11.2 Evaluasi (jalankan setelah kolom human_gold_rating terisi lengkap dan pelatihan selesai)

In [11]:
gold_result, gold_effect = gold_test.evaluate_on_gold(ctx_main, SEEDS_MAIN)
display(gold_result); display(gold_effect)

[OK] Hash sidecar uji emas cocok dengan data uji run ini (n dasar = 2856).
Baris 'tidak dapat ditentukan dari teks': 0 (0.00%)
Evaluasi enam skenario pada subset uji emas (n=93).
                Model  n_gold  n_seed      MAE     RMSE  accuracy  off_by_one      QWK
     M4_Baseline_CORN      93       3 0.379928 0.649742  0.641577    0.978495 0.887853
M6_CleanedSevere_CORN      93       3 0.408602 0.655583  0.602151    0.989247 0.888043
  M3_CleanedSevere_CE      93       3 0.419355 0.698177  0.609319    0.978495 0.871273
  M5_CleanedHard_CORN      93       3 0.430108 0.717914  0.609319    0.964158 0.866780
    M2_CleanedHard_CE      93       3 0.462366 0.744945  0.580645    0.960573 0.860594
       M1_Baseline_CE      93       3 0.491039 0.794830  0.577061    0.935484 0.846660
                 hypothesis               model_a             model_b  mean_diff  ci_95_low  ci_95_high         arah  ci_melewati_nol
          H1_CORN_vs_CE_raw      M4_Baseline_CORN      M1_Baseline_CE  -0.1111

,Model,n_gold,n_seed,MAE,RMSE,accuracy,off_by_one,QWK
0,M4_Baseline_CORN,93,3,0.379928,0.649742,0.641577,0.978495,0.887853
1,M6_CleanedSevere_CORN,93,3,0.408602,0.655583,0.602151,0.989247,0.888043
2,M3_CleanedSevere_CE,93,3,0.419355,0.698177,0.609319,0.978495,0.871273
3,M5_CleanedHard_CORN,93,3,0.430108,0.717914,0.609319,0.964158,0.866780
4,M2_CleanedHard_CE,93,3,0.462366,0.744945,0.580645,0.960573,0.860594
5,M1_Baseline_CE,93,3,0.491039,0.794830,0.577061,0.935484,0.846660


,hypothesis,model_a,model_b,mean_diff,ci_95_low,ci_95_high,arah,ci_melewati_nol
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,-0.111111,-0.189964,-0.032258,A lebih baik,False
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.028674,-0.039427,0.096774,B lebih baik,True
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.050179,-0.028674,0.121864,B lebih baik,True
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.021505,-0.096774,0.053763,A lebih baik,True
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,-0.071685,-0.154122,0.010753,A lebih baik,True


In [12]:
try:
    gold_kappa = gold_test.compute_gold_kappa(ctx_main)
except (FileNotFoundError, ValueError) as e:
    print("[PERINGATAN] Kappa uji emas belum dapat dihitung:", e)

Kappa berbobot kuadratik antar-penilai uji emas (n=28): 0.9598


In [13]:
selfcheck.check_gold(ctx_main)

Pemeriksaan mandiri: 11. Uji emas
  [OK] ada: gold_test_evaluation.csv
  [OK] ada: gold_test_effect_sizes.csv
  [OK] ada: gold_test_undetermined_summary.csv
[OK] 11. Uji emas: seluruh pemeriksaan umum lulus.


True

## 12. Sensitivitas jumlah fold K

**Tujuan.** Memeriksa sensitivitas K = 3, 5, dan 10 pada proxy P4 sebagai pelengkap konfirmatif atas keputusan apriori K = 5, bukan pencarian nilai optimum. K = 5 berasal dari proses utama pada eksekusi ini (tidak dilatih dua kali).

**Rujukan proposal.** Subbab 3.5.

**Masukan.** runs/split_seed42/data/split_train.csv dan OOF P4 K = 5.

**Keluaran.** runs/split_seed42/k_sensitivity/{k_sensitivity_table, k_sensitivity_qwk_pairs}.csv, berkas OOF dan fold K = 3 dan K = 10.

**Perkiraan waktu GPU.** sekitar 130 menit (K = 3 dan K = 10) (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Tabel melaporkan QWK, MAE, akurasi, off-by-one, ECE kumulatif rata-rata, metode filter terpilih, jumlah baris ter-flag per metode, dan indeks Jaccard himpunan ter-flag terhadap K = 5. TAMBAHAN di luar kalimat proposal: CI bootstrap berpasangan untuk selisih QWK antar K, supaya klaim 'selisih kecil dan tidak monoton' memiliki ukuran ketidakpastian.

**Peringatan.** Komponen ini ditolak untuk split selain split utama.

In [16]:
df_train_main = pd.read_csv(ctx_main.split_file("train"))
print(ctx_main.root, len(df_train_main))

D:\SKRIPSI\SKRIPSI_CORN_CL_FINAL\runs\split_seed42 9949


In [17]:
ev_p4 = ablation.run_proxy_evaluation(ctx_main, df_train_main, 3, config.PROXY_CV_FOLDS, ctx_main.diagnostics_dir)
k_table, k_pairs = sensitivity_k.run_k_sensitivity(ctx_main, df_train_main, ev_p4)
display(k_table); display(k_pairs)


=== P4 finetuned_corn (K=5) ===
[OK] OOF finetuned_corn K=5 sudah ada, dimuat.
   confident_learning: 4757 baris (47.81%), selisih terhadap estimasi: 723  <-- DIPILIH
   prune_by_noise_rate: 4250 baris (42.72%), selisih terhadap estimasi: 1230
   estimasi teoretis (off_diagonal_calibrated): 5480 baris
Kualitas proxy: acc=0.4593 MAE=0.7900 off-by-1=0.8164 QWK=0.6754

=== P4 finetuned_corn (K=3) ===
   [finetuned_corn] K=3 fold 1/3 (train=6632, val=3317)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6286.37it/s]


      Epoch 1/3 - Loss: 0.5117
      Epoch 2/3 - Loss: 0.4437
      Epoch 3/3 - Loss: 0.3709
      Temperature scaling: T=1.2431 (NLL 1.3705 -> 1.3095)
   [finetuned_corn] K=3 fold 2/3 (train=6633, val=3316)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7383.42it/s]


      Epoch 1/3 - Loss: 0.5103
      Epoch 2/3 - Loss: 0.4432
      Epoch 3/3 - Loss: 0.3779
      Temperature scaling: T=1.2711 (NLL 1.4232 -> 1.3459)
   [finetuned_corn] K=3 fold 3/3 (train=6633, val=3316)...


...

      Epoch 1/3 - Loss: 0.5060
      Epoch 2/3 - Loss: 0.4441
      Epoch 3/3 - Loss: 0.3772
      Temperature scaling: T=1.2397 (NLL 1.3758 -> 1.3216)
   [finetuned_corn] K=10 fold 10/10 (train=8955, val=994)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5217.81it/s]


      Epoch 1/3 - Loss: 0.5033
      Epoch 2/3 - Loss: 0.4443
      Epoch 3/3 - Loss: 0.3775
      Temperature scaling: T=1.1792 (NLL 1.3213 -> 1.2934)
   Temperature per fold: ['1.226', '1.204', '1.249', '1.201', '1.230', '1.218', '1.206', '1.272', '1.240', '1.179'] (rata-rata 1.223)
   confident_learning: 4892 baris (49.17%), selisih terhadap estimasi: 602  <-- DIPILIH
   prune_by_noise_rate: 4369 baris (43.91%), selisih terhadap estimasi: 1125
   estimasi teoretis (off_diagonal_calibrated): 5494 baris
Kualitas proxy: acc=0.4540 MAE=0.7855 off-by-1=0.8221 QWK=0.6772


,K,qwk,mae,accuracy,off_by_one,ece_kumulatif_rantai_rata,metode_filter_terpilih,n_flag_confident_learning,n_flag_prune_by_noise_rate,n_estimasi_teoretis,n_flag_metode_terpilih,jaccard_vs_K5
0,3,0.675505,0.787918,0.462258,0.817670,0.048824,confident_learning,4821,4322,5419,4821,0.602476
1,5,0.675357,0.790029,0.459343,0.816363,0.042001,confident_learning,4757,4250,5480,4757,1.000000
2,10,0.677166,0.785506,0.454015,0.822093,0.037246,confident_learning,4892,4369,5494,4892,0.592507


,K_x,K_y,qwk_diff_x_minus_y,ci_95_low,ci_95_high,ci_melewati_nol
0,3,5,0.000148,-0.008569,0.008992,True
1,3,10,-0.001662,-0.011164,0.007506,True
2,5,10,-0.001809,-0.011343,0.007444,True


In [18]:
selfcheck.check_k_sensitivity(ctx_main)

Pemeriksaan mandiri: 12. Sensitivitas K
  [OK] ada: k_sensitivity_table.csv
  [OK] ada: k_sensitivity_qwk_pairs.csv
  [OK] nilai K pada tabel: [3, 5, 10]
[OK] 12. Sensitivitas K: seluruh pemeriksaan umum lulus.


True

## 13. Sensitivitas partisi data

**Tujuan.** Menjawab permintaan dosen pembimbing agar kesimpulan tidak bergantung pada satu undian split. Selain split utama (seed partisi 42), dijalankan dua split acak tambahan (seed partisi 123 dan 2024) dengan rasio 70:10:20, stratifikasi dan pengelompokan yang sama. Total tiga skenario split.

**Rujukan proposal.** Subbab 3.4 dan 3.12.

**Masukan.** shared/processed/reviews_clean.csv.

**Keluaran.** runs/split_seed123/ dan runs/split_seed2024/ (struktur seragam dengan split utama), serta shared/results/sensitivity_partisi/{tabel_panjang_semua_split, ringkasan_konsistensi, wilcoxon_berdampingan_per_split}.csv.

**Perkiraan waktu GPU.** per split tambahan: P4 lima fold sekitar 50 menit dan 18 sesi pelatihan sekitar 8 sampai 12 jam (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Lihat aturan konsistensi di bawah. Hasil Wilcoxon per split dilaporkan berdampingan dan tidak digabung.

**Peringatan.** Ablasi P1 sampai P3, sensitivitas K, validasi manusia, dan uji emas hanya untuk split utama karena anotasi manual tidak dapat diulang per split. Kode menolak menjalankan komponen tersebut pada split lain.

### Dasar rancangan dan aturan konsistensi (ditetapkan SEBELUM menjalankan sel sensitivitas)

**Mengapa seed partisi, bukan rasio.** Keberatan yang dijawab adalah apakah kesimpulan stabil terhadap undian partisi.
Mengubah seed hanya mengubah komposisi sampel. Mengubah rasio mengubah ukuran data latih dan data uji sekaligus, sehingga
selisih hasil tidak dapat diatribusikan pada satu penyebab, dan metrik antar rasio tidak sebanding.

**Pipeline per split (identik).** Pembuatan split, proxy P4 lima fold dengan temperature scaling, Confident learning,
resolusi konflik, tiga varian data, enam skenario M1 sampai M6 dengan tiga seed bobot (18 sesi), uji Wilcoxon dengan koreksi
Holm-Bonferroni (lima hipotesis), effect size MAE dengan CI bootstrap, QWK ensemble dengan CI bootstrap, dan QWK per seed
dengan CI bootstrap. Opsi hemat (satu seed bobot per split, sakelar HEMAT_SPLIT_SENSITIVITAS) tersedia dan, bila dipakai,
wajib dinyatakan sebagai keterbatasan.

**Arah.** Untuk MAE, selisih (A minus B) negatif berarti model A lebih baik. Untuk QWK, selisih positif berarti model A lebih baik.

**Aturan konsistensi.** Suatu temuan dinyatakan KONSISTEN bila arahnya sama dan CI 95 persen bootstrap tidak melewati nol pada
sekurangnya dua dari tiga split. Ia dinyatakan TIDAK KONSISTEN bila arahnya berbalik atau CI melewati nol pada sekurangnya dua
split. Penerapan operasional: konsisten bila arah titik taksiran seragam pada ketiga split dan sekurangnya dua CI tidak melewati nol;
selain itu tidak konsisten (dengan tiga split, kedua kategori saling lengkap).

**Larangan.** Semua split dilaporkan seluruhnya, tidak ada pemilihan split berdasarkan hasil, dan p-value tidak digabung antar split.
Data uji tiap split berasal dari korpus yang sama dan saling tumpang tindih, sehingga ketiganya bukan replikasi independen.

**Batasan.** Validasi manusia dan uji emas hanya untuk split utama.

In [19]:
SPLITS_TAMBAHAN = [s for s in config.SPLIT_SEEDS if s != config.MAIN_SPLIT_SEED]
ctx_sens = {s: config.RunPaths(s) for s in SPLITS_TAMBAHAN}
seeds_sens = {s: pipeline.weight_seeds_for(s, hemat=HEMAT_SPLIT_SENSITIVITAS) for s in SPLITS_TAMBAHAN}
print("Split tambahan:", SPLITS_TAMBAHAN, "| seed bobot:", seeds_sens)
if HEMAT_SPLIT_SENSITIVITAS:
    print("[PERINGATAN] OPSI HEMAT aktif. Wajib dinyatakan sebagai keterbatasan.")

Split tambahan: [123, 2024] | seed bobot: {123: [42, 123, 2024], 2024: [42, 123, 2024]}


### 13.1 Pembuatan split tambahan

In [20]:
df_train_sens = {}
for s, c in ctx_sens.items():
    tr, va, te = data_split.create_splits(c)
    df_train_sens[s] = tr
    pipeline.register_run_mode(c, seeds_sens[s], HEMAT_SPLIT_SENSITIVITAS)
    selfcheck.check_split(c)

Split seed 123: train=9950, val=1423, test=2848 (dari 14221 baris, 13979 teks unik)
Baris dalam grup teks duplikat: 393 (2.76%)
Overlap lintas partisi (harus nol): {'overlap_teks_train_val': 0, 'overlap_teks_train_test': 0, 'overlap_teks_val_test': 0, 'overlap_review_id_train_val': 0, 'overlap_review_id_train_test': 0, 'overlap_review_id_val_test': 0}
Pemeriksaan mandiri: 5. Pembagian data (split 123)
  [OK] split_summary.csv ada (split 123)
  [OK] jumlah train + val + test sama dengan total
  [OK] overlap_teks_train_val = 0 (harus 0)
  [OK] overlap_teks_train_test = 0 (harus 0)
  [OK] overlap_teks_val_test = 0 (harus 0)
  [OK] overlap_review_id_train_val = 0 (harus 0)
  [OK] overlap_review_id_train_test = 0 (harus 0)
  [OK] overlap_review_id_val_test = 0 (harus 0)
  [OK] berkas split train terverifikasi
  [OK] berkas split val terverifikasi
  [OK] berkas split test terverifikasi
[OK] 5. Pembagian data (split 123): seluruh pemeriksaan umum lulus.
Split seed 2024: train=9968, val=1410, 

### 13.2 Proxy P4 lima fold, Confident learning, dan tiga varian data per split tambahan

In [21]:
out_p4_sens = {}
for s, c in ctx_sens.items():
    out_p4_sens[s] = pipeline.step_p4(c, df_train_sens[s])
    display(pd.Series(out_p4_sens[s]["variant_stats"], name=f"split {s}"))
    selfcheck.check_p4(c)


=== P4 finetuned_corn (K=5) ===
   [finetuned_corn] K=5 fold 1/5 (train=7960, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5485.45it/s]


      Epoch 1/3 - Loss: 0.5070
      Epoch 2/3 - Loss: 0.4493
      Epoch 3/3 - Loss: 0.3828
      Temperature scaling: T=1.2267 (NLL 1.2987 -> 1.2458)
   [finetuned_corn] K=5 fold 2/5 (train=7960, val=1990)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8281.41it/s]


      Epoch 1/3 - Loss: 0.5091
      Epoch 2/3 - Loss: 0.4429
      Epoch 3/3 - Loss: 0.3773
      Temperature scaling: T=1.1742 (NLL 1.3152 -> 1.2892)
   [finetuned_corn] K=5 fold 3/5 (train=7960, val=1990)...


...

      Epoch 1/3 - Loss: 0.5063
      Epoch 2/3 - Loss: 0.4444
      Epoch 3/3 - Loss: 0.3773
      Temperature scaling: T=1.2167 (NLL 1.3443 -> 1.2994)
   [finetuned_corn] K=5 fold 5/5 (train=7975, val=1993)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8838.62it/s]


      Epoch 1/3 - Loss: 0.5098
      Epoch 2/3 - Loss: 0.4449
      Epoch 3/3 - Loss: 0.3812
      Temperature scaling: T=1.2434 (NLL 1.4268 -> 1.3676)
   Temperature per fold: ['1.257', '1.223', '1.218', '1.217', '1.243'] (rata-rata 1.232)
   confident_learning: 4982 baris (49.98%), selisih terhadap estimasi: 611  <-- DIPILIH
   prune_by_noise_rate: 4343 baris (43.57%), selisih terhadap estimasi: 1250
   estimasi teoretis (off_diagonal_calibrated): 5593 baris
Kualitas proxy: acc=0.4463 MAE=0.8015 off-by-1=0.8105 QWK=0.6654
Distribusi rating_diff pada baris noise (sebelum resolusi):
rating_diff
1    3268
2    1351
3     259
4     104
Baris dibuang (rating minoritas dalam grup konflik): 0
Baris dibuang (seri, tidak ada mayoritas jelas)    : 296
Total dibuang                                      : 296
Resolusi konflik teks identik (pasca deteksi): 9968 -> 9672 baris
Raw 9672 | Hard-prune: buang 4845, sisa 4827 | Severity-aware: buang 1656, sisa 8016


split_seed                                     2024
n_latih_awal                                   9968
n_setelah_resolusi_konflik                     9672
n_dibuang_resolusi_konflik                      296
metode_filter_terpilih           confident_learning
n_flag_noise_sebelum_resolusi                  4982
n_raw                                          9672
n_hard                                         4827
n_severe                                       8016
n_dibuang_hard                                 4845
n_dibuang_severe                               1656
n_confident_learning                           4982
n_prune_by_noise_rate                          4343
n_estimated_theoretical                        5593
Name: split 2024, dtype: object

Pemeriksaan mandiri: 7. P4, deteksi noise, dan varian (split 2024)
  [OK] OOF P4 K=5 tersimpan
  [OK] bentuk OOF (9968, 5)
  [OK] probabilitas kelas berjumlah satu
  [OK] kumulatif rantai CORN tersimpan
  [OK] satu temperature per fold
  [OK] varian data latih 'raw' ada
  [OK] varian data latih 'hard' ada
  [OK] varian data latih 'severe' ada
  [OK] urutan ukuran hard <= severe <= raw ({'raw': 9672, 'hard': 4827, 'severe': 8016})
[OK] 7. P4, deteksi noise, dan varian (split 2024): seluruh pemeriksaan umum lulus.


### 13.3 Pelatihan enam skenario per split tambahan

In [22]:
status_123 = pipeline.step_train(ctx_sens[123], seeds_sens[123], copy_ckpt_to_drive=False)
selfcheck.check_training(ctx_sens[123])


[MULAI] M1_Baseline_CE | split 123 | seed bobot 42 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6930.55it/s]


Epoch 1/10 - Loss: 1.3521 - Val MAE: 0.7885 | QWK: 0.6763 | Off-by-1: 0.8103 | Acc: 0.4610
Epoch 2/10 - Loss: 1.1918 - Val MAE: 0.7716 | QWK: 0.6943 | Off-by-1: 0.8152 | Acc: 0.4751
Epoch 3/10 - Loss: 1.0294 - Val MAE: 0.7632 | QWK: 0.6923 | Off-by-1: 0.8257 | Acc: 0.4786
Epoch 4/10 - Loss: 0.7870 - Val MAE: 0.7744 | QWK: 0.6969 | Off-by-1: 0.8292 | Acc: 0.4610
Epoch 5/10 - Loss: 0.5095 - Val MAE: 0.7786 | QWK: 0.6808 | Off-by-1: 0.8264 | Acc: 0.4596
Epoch 6/10 - Loss: 0.2864 - Val MAE: 0.7618 | QWK: 0.6859 | Off-by-1: 0.8419 | Acc: 0.4462
Epoch 7/10 - Loss: 0.1641 - Val MAE: 0.7611 | QWK: 0.6930 | Off-by-1: 0.8313 | Acc: 0.4561
Epoch 8/10 - Loss: 0.0961 - Val MAE: 0.7540 | QWK: 0.6926 | Off-by-1: 0.8489 | Acc: 0.4469
Epoch 9/10 - Loss: 0.0587 - Val MAE: 0.7576 | QWK: 0.6901 | Off-by-1: 0.8447 | Acc: 0.4519
Epoch 10/10 - Loss: 0.0368 - Val MAE: 0.7646 | QWK: 0.6841 | Off-by-1: 0.8419 | Acc: 0.4476
[SELESAI] Uji (checkpoint epoch 4, QWK val 0.6969): MAE=0.7830 | RMSE=1.2022 | Acc=0.4638

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5845.49it/s]


Epoch 1/10 - Loss: 1.3719 - Val MAE: 0.8236 | QWK: 0.6840 | Off-by-1: 0.7948 | Acc: 0.4596
Epoch 2/10 - Loss: 1.2015 - Val MAE: 0.7899 | QWK: 0.6898 | Off-by-1: 0.8074 | Acc: 0.4765
Epoch 3/10 - Loss: 1.0337 - Val MAE: 0.7294 | QWK: 0.7033 | Off-by-1: 0.8440 | Acc: 0.4701
Epoch 4/10 - Loss: 0.7934 - Val MAE: 0.7442 | QWK: 0.7006 | Off-by-1: 0.8517 | Acc: 0.4631
Epoch 5/10 - Loss: 0.5137 - Val MAE: 0.8067 | QWK: 0.6628 | Off-by-1: 0.8152 | Acc: 0.4357
Epoch 6/10 - Loss: 0.2771 - Val MAE: 0.8110 | QWK: 0.6586 | Off-by-1: 0.8159 | Acc: 0.4455
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7033): MAE=0.7584 | RMSE=1.1578 | Acc=0.4656 | Off-by-1=0.8294 | QWK=0.6810
   Progres sesi pelatihan: 2/18

[MULAI] M1_Baseline_CE | split 123 | seed bobot 2024 | loss CE


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4945.82it/s]


Epoch 1/10 - Loss: 0.4924 - Val MAE: 0.7590 | QWK: 0.7097 | Off-by-1: 0.8292 | Acc: 0.4610
Epoch 2/10 - Loss: 0.3746 - Val MAE: 0.6978 | QWK: 0.7299 | Off-by-1: 0.8623 | Acc: 0.4793
Epoch 3/10 - Loss: 0.3080 - Val MAE: 0.7203 | QWK: 0.7164 | Off-by-1: 0.8595 | Acc: 0.4631
Epoch 4/10 - Loss: 0.2338 - Val MAE: 0.7259 | QWK: 0.7137 | Off-by-1: 0.8686 | Acc: 0.4441
Epoch 5/10 - Loss: 0.1608 - Val MAE: 0.7456 | QWK: 0.7103 | Off-by-1: 0.8559 | Acc: 0.4427
   Early stopping di epoch 5 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.7299): MAE=0.7223 | RMSE=1.0973 | Acc=0.4691 | Off-by-1=0.8501 | QWK=0.7136
   Progres sesi pelatihan: 17/18

[MULAI] M6_CleanedSevere_CORN | split 123 | seed bobot 2024 | loss CORN


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8083.00it/s]


Epoch 1/10 - Loss: 0.4841 - Val MAE: 0.7639 | QWK: 0.7024 | Off-by-1: 0.8587 | Acc: 0.4146
Epoch 2/10 - Loss: 0.3718 - Val MAE: 0.7372 | QWK: 0.7154 | Off-by-1: 0.8559 | Acc: 0.4505
Epoch 3/10 - Loss: 0.3029 - Val MAE: 0.7287 | QWK: 0.7200 | Off-by-1: 0.8517 | Acc: 0.4659
Epoch 4/10 - Loss: 0.2294 - Val MAE: 0.7168 | QWK: 0.7198 | Off-by-1: 0.8623 | Acc: 0.4596
Epoch 5/10 - Loss: 0.1586 - Val MAE: 0.7175 | QWK: 0.7232 | Off-by-1: 0.8658 | Acc: 0.4554
Epoch 6/10 - Loss: 0.0972 - Val MAE: 0.7231 | QWK: 0.7166 | Off-by-1: 0.8630 | Acc: 0.4561
Epoch 7/10 - Loss: 0.0646 - Val MAE: 0.7252 | QWK: 0.7136 | Off-by-1: 0.8580 | Acc: 0.4603
   Early stopping di epoch 7 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 5, QWK val 0.7232): MAE=0.7251 | RMSE=1.0912 | Acc=0.4568 | Off-by-1=0.8610 | QWK=0.7161
   Progres sesi pelatihan: 18/18
Pemeriksaan mandiri: 9. Pelatihan (split 123)
  [OK] run_mode.json ada
  [OK] sesi pelatihan lengkap 18/18
[OK] 9. Pelatihan (split 123): seluru

True

In [23]:
status_2024 = pipeline.step_train(ctx_sens[2024], seeds_sens[2024], copy_ckpt_to_drive=False)
selfcheck.check_training(ctx_sens[2024])


[MULAI] M1_Baseline_CE | split 2024 | seed bobot 42 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5527.74it/s]


Epoch 1/10 - Loss: 1.3525 - Val MAE: 0.7901 | QWK: 0.6895 | Off-by-1: 0.8014 | Acc: 0.4674
Epoch 2/10 - Loss: 1.1890 - Val MAE: 0.7738 | QWK: 0.6999 | Off-by-1: 0.8277 | Acc: 0.4610
Epoch 3/10 - Loss: 1.0294 - Val MAE: 0.7773 | QWK: 0.6994 | Off-by-1: 0.8149 | Acc: 0.4617
Epoch 4/10 - Loss: 0.7958 - Val MAE: 0.7355 | QWK: 0.6976 | Off-by-1: 0.8617 | Acc: 0.4447
Epoch 5/10 - Loss: 0.5393 - Val MAE: 0.7801 | QWK: 0.6800 | Off-by-1: 0.8298 | Acc: 0.4511
Epoch 6/10 - Loss: 0.3145 - Val MAE: 0.7801 | QWK: 0.6679 | Off-by-1: 0.8468 | Acc: 0.4199
Epoch 7/10 - Loss: 0.1742 - Val MAE: 0.7610 | QWK: 0.6763 | Off-by-1: 0.8489 | Acc: 0.4411
   Early stopping di epoch 7 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 2, QWK val 0.6999): MAE=0.7946 | RMSE=1.2305 | Acc=0.4752 | Off-by-1=0.8051 | QWK=0.6706
   Progres sesi pelatihan: 1/18

[MULAI] M1_Baseline_CE | split 2024 | seed bobot 123 | loss CE


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9287.18it/s]


Epoch 1/10 - Loss: 1.3637 - Val MAE: 0.7539 | QWK: 0.6981 | Off-by-1: 0.8184 | Acc: 0.4674
Epoch 2/10 - Loss: 1.1993 - Val MAE: 0.7652 | QWK: 0.7031 | Off-by-1: 0.8291 | Acc: 0.4681
Epoch 3/10 - Loss: 1.0289 - Val MAE: 0.7184 | QWK: 0.7170 | Off-by-1: 0.8539 | Acc: 0.4730
Epoch 4/10 - Loss: 0.7902 - Val MAE: 0.7461 | QWK: 0.6975 | Off-by-1: 0.8475 | Acc: 0.4539
Epoch 5/10 - Loss: 0.5055 - Val MAE: 0.7468 | QWK: 0.6880 | Off-by-1: 0.8383 | Acc: 0.4695
Epoch 6/10 - Loss: 0.2984 - Val MAE: 0.7603 | QWK: 0.6740 | Off-by-1: 0.8362 | Acc: 0.4624
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7170): MAE=0.7281 | RMSE=1.1256 | Acc=0.4780 | Off-by-1=0.8463 | QWK=0.6988
   Progres sesi pelatihan: 2/18

[MULAI] M1_Baseline_CE | split 2024 | seed bobot 2024 | loss CE


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8731.83it/s]


Epoch 1/10 - Loss: 0.4890 - Val MAE: 0.7248 | QWK: 0.7129 | Off-by-1: 0.8574 | Acc: 0.4582
Epoch 2/10 - Loss: 0.3662 - Val MAE: 0.7376 | QWK: 0.6992 | Off-by-1: 0.8546 | Acc: 0.4433
Epoch 3/10 - Loss: 0.2920 - Val MAE: 0.7092 | QWK: 0.7202 | Off-by-1: 0.8539 | Acc: 0.4830
Epoch 4/10 - Loss: 0.2213 - Val MAE: 0.7184 | QWK: 0.7137 | Off-by-1: 0.8617 | Acc: 0.4603
Epoch 5/10 - Loss: 0.1571 - Val MAE: 0.7184 | QWK: 0.7089 | Off-by-1: 0.8624 | Acc: 0.4631
Epoch 6/10 - Loss: 0.0994 - Val MAE: 0.7043 | QWK: 0.7085 | Off-by-1: 0.8695 | Acc: 0.4667
Epoch 7/10 - Loss: 0.0639 - Val MAE: 0.7085 | QWK: 0.7152 | Off-by-1: 0.8660 | Acc: 0.4681
Epoch 8/10 - Loss: 0.0441 - Val MAE: 0.7106 | QWK: 0.7114 | Off-by-1: 0.8638 | Acc: 0.4674
Epoch 9/10 - Loss: 0.0281 - Val MAE: 0.7184 | QWK: 0.7078 | Off-by-1: 0.8631 | Acc: 0.4589
   Early stopping di epoch 9 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 3, QWK val 0.7202): MAE=0.7295 | RMSE=1.1073 | Acc=0.4657 | Off-by-1=0.8491 | QWK=0.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6585.87it/s]


Epoch 1/10 - Loss: 0.4796 - Val MAE: 0.7199 | QWK: 0.7234 | Off-by-1: 0.8511 | Acc: 0.4730
Epoch 2/10 - Loss: 0.3662 - Val MAE: 0.7227 | QWK: 0.7221 | Off-by-1: 0.8638 | Acc: 0.4532
Epoch 3/10 - Loss: 0.2930 - Val MAE: 0.7007 | QWK: 0.7192 | Off-by-1: 0.8617 | Acc: 0.4780
Epoch 4/10 - Loss: 0.2168 - Val MAE: 0.7333 | QWK: 0.6992 | Off-by-1: 0.8539 | Acc: 0.4525
Epoch 5/10 - Loss: 0.1483 - Val MAE: 0.7121 | QWK: 0.7195 | Off-by-1: 0.8681 | Acc: 0.4560
Epoch 6/10 - Loss: 0.0978 - Val MAE: 0.7106 | QWK: 0.7117 | Off-by-1: 0.8638 | Acc: 0.4589
   Early stopping di epoch 6 (MAE validasi tidak membaik 3 kali)
[SELESAI] Uji (checkpoint epoch 1, QWK val 0.7234): MAE=0.7594 | RMSE=1.1517 | Acc=0.4604 | Off-by-1=0.8340 | QWK=0.6969
   Progres sesi pelatihan: 18/18
Pemeriksaan mandiri: 9. Pelatihan (split 2024)
  [OK] run_mode.json ada
  [OK] sesi pelatihan lengkap 18/18
[OK] 9. Pelatihan (split 2024): seluruh pemeriksaan umum lulus.


True

### 13.4 Analisis per split tambahan

In [24]:
out_sig_sens = {}
for s, c in ctx_sens.items():
    out_sig_sens[s] = pipeline.step_analysis(c, seeds_sens[s])
    print(f"Split {s}")
    display(out_sig_sens[s]["final"]); display(out_sig_sens[s]["sig"])
    selfcheck.check_significance(c)

Split 123


,Model,n_seed,MAE_mean,MAE_std,RMSE_mean,RMSE_std,Acc_mean,Acc_std,Off-by-1_mean,Off-by-1_std,QWK_mean,QWK_std
0,M4_Baseline_CORN,3,0.724836,0.011259,1.091553,0.018509,0.459153,0.010015,0.857795,0.001744,0.713175,0.003294
1,M6_CleanedSevere_CORN,3,0.725070,0.002294,1.092991,0.003037,0.459153,0.007356,0.858380,0.006016,0.713484,0.002212
2,M5_CleanedHard_CORN,3,0.738881,0.009167,1.113928,0.012306,0.453769,0.004379,0.854986,0.006975,0.707137,0.003103
3,M2_CleanedHard_CE,3,0.740051,0.007884,1.127947,0.009457,0.460908,0.004635,0.851709,0.002664,0.708587,0.002903
4,M3_CleanedSevere_CE,3,0.744148,0.006621,1.128372,0.025540,0.459036,0.008612,0.847144,0.007140,0.705799,0.002060
5,M1_Baseline_CE,3,0.759129,0.019215,1.158721,0.035126,0.461962,0.003955,0.836142,0.016757,0.690694,0.010048


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.724836,0.759129,0.000014,0.000071,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.725070,0.724836,0.984152,0.984152,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.738881,0.724836,0.018673,0.074692,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.725070,0.738881,0.024937,0.074811,Tidak
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.744148,0.759129,0.059213,0.118426,Tidak


Pemeriksaan mandiri: 10. Uji signifikansi (split 123)
  [OK] ada: final_results_table.csv
  [OK] ada: significance_test.csv
  [OK] ada: effect_sizes_mae.csv
  [OK] ada: qwk_ensemble_effect.csv
  [OK] ada: qwk_per_seed_effect.csv
  [OK] ada: tabel_panjang_efek.csv
  [OK] jumlah hipotesis pada tabel = 5
[OK] 10. Uji signifikansi (split 123): seluruh pemeriksaan umum lulus.
Split 2024


,Model,n_seed,MAE_mean,MAE_std,RMSE_mean,RMSE_std,Acc_mean,Acc_std,Off-by-1_mean,Off-by-1_std,QWK_mean,QWK_std
0,M4_Baseline_CORN,3,0.723180,0.001883,1.091527,0.005758,0.463243,0.006473,0.853676,0.006970,0.708497,0.004569
1,M5_CleanedHard_CORN,3,0.737249,0.009650,1.125487,0.026838,0.468754,0.007772,0.842185,0.009833,0.697608,0.002631
2,M2_CleanedHard_CE,3,0.739125,0.009281,1.120845,0.011234,0.463478,0.003704,0.842889,0.006088,0.699412,0.004809
3,M3_CleanedSevere_CE,3,0.740532,0.012915,1.125713,0.030234,0.465471,0.003920,0.840544,0.007491,0.701118,0.002610
4,M6_CleanedSevere_CORN,3,0.746395,0.012509,1.121747,0.021186,0.455388,0.011072,0.843944,0.007048,0.699090,0.008392
5,M1_Baseline_CE,3,0.768789,0.029111,1.183623,0.043558,0.470864,0.008206,0.824247,0.016928,0.685859,0.011633


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.723180,0.768789,4.225188e-07,0.000002,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.746395,0.723180,2.901246e-04,0.000924,Ya
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.737249,0.723180,9.823044e-02,0.196461,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.746395,0.737249,2.764471e-01,0.276447,Tidak
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.740532,0.768789,2.309831e-04,0.000924,Ya


Pemeriksaan mandiri: 10. Uji signifikansi (split 2024)
  [OK] ada: final_results_table.csv
  [OK] ada: significance_test.csv
  [OK] ada: effect_sizes_mae.csv
  [OK] ada: qwk_ensemble_effect.csv
  [OK] ada: qwk_per_seed_effect.csv
  [OK] ada: tabel_panjang_efek.csv
  [OK] jumlah hipotesis pada tabel = 5
[OK] 10. Uji signifikansi (split 2024): seluruh pemeriksaan umum lulus.


### 13.5 Ringkasan konsistensi (tanpa menggabungkan p-value)

In [25]:
long_all, consistency_table, wilcoxon_wide = pipeline.consistency_summary()
display(consistency_table)
display(wilcoxon_wide)
display(long_all)

,hypothesis,metric,n_split,arah_seragam,n_split_ci_tidak_melewati_nol,n_split_ci_melewati_nol,putusan,arah_split42,ci_excl_nol_split42,arah_split123,ci_excl_nol_split123,arah_split2024,ci_excl_nol_split2024,arah_dominan
0,H1_CORN_vs_CE_raw,MAE,3,True,3,0,KONSISTEN,A,True,A,True,A,True,A lebih baik
1,H2_SeverityAware_vs_Base,MAE,3,False,1,2,TIDAK KONSISTEN,A,False,B,False,B,True,tidak seragam
2,H3_HardPrune_vs_Base,MAE,3,True,1,2,TIDAK KONSISTEN,B,False,B,True,B,False,B lebih baik
3,H4_SeverityAware_vs_Hard,MAE,3,False,2,1,TIDAK KONSISTEN,A,True,A,True,B,False,tidak seragam
4,H5_SeverityAware_vs_Base_CE,MAE,3,True,2,1,KONSISTEN,A,True,A,False,A,True,A lebih baik
5,H1_CORN_vs_CE_raw,QWK_ensemble,3,True,2,1,KONSISTEN,A,False,A,True,A,True,A lebih baik
6,H2_SeverityAware_vs_Base,QWK_ensemble,3,True,1,2,TIDAK KONSISTEN,B,False,B,False,B,True,B lebih baik
7,H3_HardPrune_vs_Base,QWK_ensemble,3,True,2,1,KONSISTEN,B,False,B,True,B,True,B lebih baik
8,H4_SeverityAware_vs_Hard,QWK_ensemble,3,True,1,2,TIDAK KONSISTEN,A,False,A,True,A,False,A lebih baik
9,H5_SeverityAware_vs_Base_CE,QWK_ensemble,3,True,0,3,TIDAK KONSISTEN,A,False,A,False,A,False,A lebih baik


,hypothesis,wilcoxon_p_holm_split42,wilcoxon_p_holm_split123,wilcoxon_p_holm_split2024,wilcoxon_signifikan_split42,wilcoxon_signifikan_split123,wilcoxon_signifikan_split2024
0,H1_CORN_vs_CE_raw,0.000003,0.000071,0.000002,Ya,Ya,Ya
1,H2_SeverityAware_vs_Base,0.368988,0.984152,0.000924,Tidak,Tidak,Ya
2,H3_HardPrune_vs_Base,0.149265,0.074692,0.196461,Tidak,Tidak,Tidak
3,H4_SeverityAware_vs_Hard,0.039302,0.074811,0.276447,Ya,Tidak,Tidak
4,H5_SeverityAware_vs_Base_CE,0.001209,0.118426,0.000924,Ya,Tidak,Ya


,split_seed,hypothesis,metric,model_a,model_b,diff,ci_low,ci_high,n_weight_seeds,wilcoxon_p_raw,wilcoxon_p_holm,wilcoxon_signifikan
0,42,H1_CORN_vs_CE_raw,MAE,M4_Baseline_CORN,M1_Baseline_CE,-0.043534,-0.058476,-0.028942,3,6.037154e-07,0.000003,Ya
1,42,H2_SeverityAware_vs_Base,MAE,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.004785,-0.017857,0.007703,3,3.689877e-01,0.368988,Tidak
2,42,H3_HardPrune_vs_Base,MAE,M5_CleanedHard_CORN,M4_Baseline_CORN,0.012605,-0.001401,0.026964,3,7.463251e-02,0.149265,Tidak
3,42,H4_SeverityAware_vs_Hard,MAE,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,-0.017390,-0.030932,-0.004202,3,1.310063e-02,0.039302,Ya
4,42,H5_SeverityAware_vs_Base_CE,MAE,M3_CleanedSevere_CE,M1_Baseline_CE,-0.033030,-0.047855,-0.018201,3,3.021414e-04,0.001209,Ya
5,42,H1_CORN_vs_CE_raw,QWK_ensemble,M4_Baseline_CORN,M1_Baseline_CE,0.005518,-0.004627,0.015110,3,NaN,NaN,NaN
6,42,H2_SeverityAware_vs_Base,QWK_ensemble,M6_CleanedSevere_CORN,M4_Baseline_CORN,-0.000120,-0.008865,0.008112,3,NaN,NaN,NaN
7,42,H3_HardPrune_vs_Base,QWK_ensemble,M5_CleanedHard_CORN,M4_Baseline_CORN,-0.008166,-0.018161,0.001857,3,NaN,NaN,NaN
8,42,H4_SeverityAware_vs_Hard,QWK_ensemble,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.008046,-0.000908,0.016974,3,NaN,NaN,NaN
9,42,H5_SeverityAware_vs_Base_CE,QWK_ensemble,M3_CleanedSevere_CE,M1_Baseline_CE,0.002447,-0.007614,0.012159,3,NaN,NaN,NaN


In [26]:
selfcheck.check_partition_sensitivity()

Pemeriksaan mandiri: 13. Sensitivitas partisi
  [OK] split 42: prediksi lengkap 18/18
  [OK] split 42: tabel panjang efek ada
  [OK] split 123: prediksi lengkap 18/18
  [OK] split 123: tabel panjang efek ada
  [OK] split 2024: prediksi lengkap 18/18
  [OK] split 2024: tabel panjang efek ada
  [OK] ringkasan konsistensi ada
[OK] 13. Sensitivitas partisi: seluruh pemeriksaan umum lulus.


True

In [27]:
bk = drive_io.export_backup_zip(config.DRIVE_ROOT, f"cadangan_bagian13_{RUN_TAG}.zip")
print(bk)
drive_io.download_if_colab(bk["zip_path"])

{'zip_path': 'D:\\SKRIPSI\\lokal\\cadangan_bagian13_20261004_060807.zip', 'n_files': 293, 'bytes': 25677117, 'sha256': '296ea00b19280e52ddba314325387355908ef4577cbe347f612081964c999b99'}
(Bukan Colab) berkas cadangan ada di: D:\SKRIPSI\lokal\cadangan_bagian13_20261004_060807.zip


## 14. Ringkasan akhir

**Tujuan.** Memetakan setiap berkas keluaran ke tabel atau subbab proposal dan bagian Bab IV yang akan memakainya, serta menampilkan tabel kunci dari berkas di Drive (bukan dari variabel sesi).

**Rujukan proposal.** Seluruh Bab III.

**Masukan.** Seluruh berkas keluaran.

**Keluaran.** Tabel peta keluaran (ditampilkan).

**Perkiraan waktu GPU.** kurang dari 1 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Kolom 'ada' harus bernilai ya untuk seluruh berkas yang relevan dengan run ini.

**Peringatan.** Berkas preserved/scraping_summary.csv dan berkas anotasi berasal dari luar run ini.

In [28]:
output_map = summary.build_output_map()
display(output_map)

,berkas,subbab_proposal,tabel_atau_bagian,pemakaian_bab_empat,ada
0,shared/results/corpus_verification.json,3.2,Penguncian korpus,Bab III: bukti SHA-256 dan jumlah baris,ya
1,preserved/scraping_summary.csv,3.2,Filter bahasa per aplikasi dan rating (DIPERTA...,Bab III: jumlah dan persentase ulasan digugurk...,ya
2,shared/results/quality/02_distribusi_tanggal_p...,3.2,Confound temporal,Bab IV: karakteristik dan limitasi data,ya
3,shared/results/quality/03_panjang_teks_per_rat...,3.2,Panjang teks per rating,Bab IV: validasi dugaan penetapan kuota,ya
4,shared/results/quality/04_teks_duplikat_persis...,3.2,Duplikat persis,Bab IV: indikasi template atau spam,ya
5,shared/results/quality/05_konflik_sebaran_seli...,"3.2, 3.3",Teks identik rating berbeda,Bab IV: ambiguitas alami,ya
6,shared/results/preprocessing_summary.csv,3.3,Ringkasan praproses dan truncation,Bab III dan IV,ya
7,runs/split_seed42/data/split_summary.csv,3.4,Ringkasan split dan bukti nol overlap,Bab III: pembagian data,ya
8,runs/split_seed42/ablation/proxy_ablation_tabl...,"3.5, 3.6",Tabel 3.1 (hasil ablasi proxy),"Bab IV: ablasi P1 sampai P4, risiko sirkularitas",ya
9,runs/split_seed42/diagnostics/P4__finetuned_co...,3.6,Kualitas proxy per kelas rating asli,Bab IV: rujukan pemeriksaan confound,ya


In [29]:
for nama, path in [
    ("Tabel ablasi proxy (Tabel 3.1)", os.path.join(ctx_main.ablation_dir, "proxy_ablation_table.csv")),
    ("Hasil enam skenario (split utama)", os.path.join(ctx_main.significance_dir, "final_results_table.csv")),
    ("Uji signifikansi H1 sampai H5 (split utama)", os.path.join(ctx_main.significance_dir, "significance_test.csv")),
    ("Konsistensi lintas split", os.path.join(config.SENSITIVITY_SUMMARY_DIR, "ringkasan_konsistensi.csv")),
]:
    print("\n" + nama)
    if drive_io.is_verified(path):
        display(pd.read_csv(path))
    else:
        print("[PERINGATAN] belum ada:", path)


Tabel ablasi proxy (Tabel 3.1)


,proxy_id,label,name,desc,accuracy,mae,rmse,off_by_one,qwk,pct_flagged_noise,selected_filter_method,n_issues_confident_learning,n_issues_prune_by_noise_rate,n_issues_estimated_theoretical,temperature_rata,ece_top_label,ece_kumulatif_turunan_rata,ece_kumulatif_rantai_rata
0,0,P1,frozen_cls_lr,Embedding beku token [CLS] + Logistic Regressi...,0.428887,0.948437,1.414356,0.744195,0.609989,45.934265,confident_learning,4570,3946,5751,NaN,0.061261,0.075827,NaN
1,1,P2,frozen_meanpool_lr,Embedding beku mean-pooling + Logistic Regress...,0.429591,0.959895,1.435027,0.739974,0.599002,46.215700,confident_learning,4598,3952,5723,NaN,0.067247,0.079229,NaN
2,2,P3,finetuned_ce,"IndoBERT fine-tuned K-Fold, Cross-Entropy, tem...",0.450900,0.793346,1.197251,0.818977,0.667714,49.150668,confident_learning,4890,4372,5555,1.242123,0.084656,0.036781,NaN
3,3,P4,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN, temperature ...",0.459343,0.790029,1.203322,0.816363,0.675357,47.813851,confident_learning,4757,4250,5480,1.167567,0.079641,0.042001,0.042001



Hasil enam skenario (split utama)


,Model,n_seed,MAE_mean,MAE_std,RMSE_mean,RMSE_std,Acc_mean,Acc_std,Off-by-1_mean,Off-by-1_std,QWK_mean,QWK_std
0,M6_CleanedSevere_CORN,3,0.706933,0.009360,1.072833,0.013158,0.467670,0.002465,0.863796,0.005521,0.725430,0.006530
1,M4_Baseline_CORN,3,0.711718,0.001575,1.070558,0.005541,0.458567,0.005149,0.867880,0.002641,0.720740,0.003239
2,M3_CleanedSevere_CE,3,0.722222,0.003965,1.096073,0.008403,0.465920,0.003775,0.853641,0.004466,0.714731,0.003117
3,M5_CleanedHard_CORN,3,0.724323,0.003411,1.102261,0.002403,0.465570,0.003881,0.855976,0.001838,0.716059,0.003105
4,M2_CleanedHard_CE,3,0.734594,0.001031,1.117786,0.009310,0.463119,0.005442,0.851424,0.005282,0.710596,0.002224
5,M1_Baseline_CE,3,0.755252,0.008051,1.161259,0.010035,0.469538,0.004438,0.834034,0.003513,0.698720,0.009702



Uji signifikansi H1 sampai H5 (split utama)


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.711718,0.755252,6.037154e-07,0.000003,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.706933,0.711718,3.689877e-01,0.368988,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.724323,0.711718,7.463251e-02,0.149265,Tidak
3,H4_SeverityAware_vs_Hard,M6_CleanedSevere_CORN,M5_CleanedHard_CORN,0.706933,0.724323,1.310063e-02,0.039302,Ya
4,H5_SeverityAware_vs_Base_CE,M3_CleanedSevere_CE,M1_Baseline_CE,0.722222,0.755252,3.021414e-04,0.001209,Ya



Konsistensi lintas split


,hypothesis,metric,n_split,arah_seragam,n_split_ci_tidak_melewati_nol,n_split_ci_melewati_nol,putusan,arah_split42,ci_excl_nol_split42,arah_split123,ci_excl_nol_split123,arah_split2024,ci_excl_nol_split2024,arah_dominan
0,H1_CORN_vs_CE_raw,MAE,3,True,3,0,KONSISTEN,A,True,A,True,A,True,A lebih baik
1,H2_SeverityAware_vs_Base,MAE,3,False,1,2,TIDAK KONSISTEN,A,False,B,False,B,True,tidak seragam
2,H3_HardPrune_vs_Base,MAE,3,True,1,2,TIDAK KONSISTEN,B,False,B,True,B,False,B lebih baik
3,H4_SeverityAware_vs_Hard,MAE,3,False,2,1,TIDAK KONSISTEN,A,True,A,True,B,False,tidak seragam
4,H5_SeverityAware_vs_Base_CE,MAE,3,True,2,1,KONSISTEN,A,True,A,False,A,True,A lebih baik
5,H1_CORN_vs_CE_raw,QWK_ensemble,3,True,2,1,KONSISTEN,A,False,A,True,A,True,A lebih baik
6,H2_SeverityAware_vs_Base,QWK_ensemble,3,True,1,2,TIDAK KONSISTEN,B,False,B,False,B,True,B lebih baik
7,H3_HardPrune_vs_Base,QWK_ensemble,3,True,2,1,KONSISTEN,B,False,B,True,B,True,B lebih baik
8,H4_SeverityAware_vs_Hard,QWK_ensemble,3,True,1,2,TIDAK KONSISTEN,A,False,A,True,A,False,A lebih baik
9,H5_SeverityAware_vs_Base_CE,QWK_ensemble,3,True,0,3,TIDAK KONSISTEN,A,False,A,False,A,False,A lebih baik


## 15. Manifest hasil dan unduhan cadangan

**Tujuan.** Mencatat SHA-256 seluruh berkas hasil sebagai bukti integritas run, lalu mengunduh cadangan zip berisi seluruh CSV, JSON, dan prediksi kecil.

**Rujukan proposal.** Subbab 3.2 (prinsip penguncian) dan 3.13.

**Masukan.** Seluruh berkas di runs, shared, logs, annotations, dan preserved.

**Keluaran.** shared/results/results_manifest.csv; berkas zip lokal.

**Perkiraan waktu GPU.** kurang dari 2 menit (perkiraan perencanaan mengikuti Subbab 3.13, bukan jaminan, dan bergantung pada GPU yang dialokasikan Colab).

**Cara membaca hasil.** Manifest memuat jalur, ukuran, dan SHA-256 tiap berkas. Berkas manifest sendiri tidak ikut di dalamnya.

**Peringatan.** Jalankan hanya setelah seluruh bagian selesai. Bila berkas hasil bertambah setelah manifest ditulis, manifest lama tidak ditimpa dan eksekusi berhenti dengan pesan; hapus manifest lama secara manual bila memang perlu dibuat ulang.

In [30]:
manifest_df, manifest_res = summary.build_results_manifest()
print(manifest_res)
display(manifest_df.head(30))

{'path': 'D:\\SKRIPSI\\SKRIPSI_CORN_CL_FINAL\\shared\\results\\results_manifest.csv', 'status': 'ditulis', 'bytes': 35560, 'sha256': '378c6bc9056b1dcd6877c2854e19300d4053686d34488c38cac37ee340d09c3b'}


,berkas,bytes,sha256
0,annotations\gold_test_sample.csv,3655,5b9b07014ec4a9bcf3ea26703f6e6595d1edf67f5e46d1...
1,annotations\gold_test_sample.csv.sidecar.json,314,1a91eab57b30e1c91722a57320b77fa9c7539d65e8cef6...
2,annotations\gold_test_sample_annotator2.csv,1120,afbdf6832d86d9de29efddad15a0d1e094c96ed0e32762...
3,annotations\gold_test_sample_annotator2.csv.si...,298,8b6c5e912785942e339ef9a7753bddbdcb489c5af8d400...
4,annotations\human_validation_sample.csv,9359,b65e2543b0f2e2976add8a54deaa660255e151ffe7405f...
5,annotations\human_validation_sample.csv.sideca...,392,8092dc5416b216db6dd5919f55e501f873112bc3c2a19c...
6,annotations\human_validation_sample_annotator2...,2673,a1aabe1e57f00e686ae2bf002c28d2ece4a25adcf5a53c...
7,annotations\human_validation_sample_annotator2...,313,fbe6b4d4d1f56d1b892eca0ad9adb2b3fe92e0213b7e0b...
8,annotations\unlabelled_gold_test_sample.csv,14681,8b2435cb5e7fab2d2806a92ccd4265841016434258600c...
9,annotations\unlabelled_gold_test_sample_annota...,3859,37dd94e8149c2ce329ce377c6b6efcd6bf2560e3a8be11...


In [31]:
bk = drive_io.export_backup_zip(config.DRIVE_ROOT, f"cadangan_final_{RUN_TAG}.zip")
print(bk)
drive_io.download_if_colab(bk["zip_path"])
print("[OK] Notebook selesai. Simpan berkas zip cadangan di luar Drive.")

{'zip_path': 'D:\\SKRIPSI\\lokal\\cadangan_final_20261004_060807.zip', 'n_files': 294, 'bytes': 25691143, 'sha256': '25a3b33c7b6bcb13c9c11968d369ba323965aac26fad6f57a15e3b0edf4e0d63'}
(Bukan Colab) berkas cadangan ada di: D:\SKRIPSI\lokal\cadangan_final_20261004_060807.zip
[OK] Notebook selesai. Simpan berkas zip cadangan di luar Drive.
